# Table Transformer (DETR-R18, PubTables-1M) — DIMER E2E supervised adaptation tutorial: nutrition-table detection on product photographs, new heads vs bounded decoder unfreeze (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/table-transformer-detection-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/table-transformer-detection-pipeline/blob/main/tutorials/table_transformer_detection_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-microsoft%2Ftable--transformer--detection-ffcc4d?style=flat)](https://huggingface.co/microsoft/table-transformer-detection) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2Ftable--transformer-181717?style=flat&logo=github&logoColor=white)](https://github.com/microsoft/table-transformer) [![arXiv](https://img.shields.io/badge/arXiv-2110.00061-b31b1b.svg)](https://arxiv.org/abs/2110.00061)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** table detection on document page images (boxes labelled `table` or `table rotated`) and bounded supervised adaptation to a new box class on a new image domain — new detection heads on the frozen DETR decoder features with an optional unfreeze of the last decoder layers — measured by held-out AP@0.5 / AP@0.75 / mAP, using the pinned `microsoft/table-transformer-detection` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/table_transformer_detection_pipeline/`, at revision `1530d418406c`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `2357cbe2b5a5d1c03e54f32764f06058933b65ab` (~115 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned Table Transformer detection snapshot (safetensors, 115 MB), fetches 121 digest-pinned Open Food Facts product photographs with their nutrition-table boxes (129 MB, no credential), validates them and draws 72 / 24 / 25 training, validation and test photographs by a seeded split of whole products, runs one test photograph through the inference contract with an input manifest, a rejection probe and a `sample-sanity` evaluation report against its reference boxes, scores a fixed-box prior and the untouched checkpoint on the test split (the **zero-shot row**), trains a new class head and a copy of the box head on the frozen DETR decoder features (the **frozen policy**) and then the last two decoder layers with them (the **unfrozen policy**), selects between the two by validation DETR loss, scores the held-out split by AP@0.5 / AP@0.75 / mAP with the selected model, renders detections before and after, exports the trained tensors as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about 4 minutes of model time after the downloads; a CUDA runtime is used automatically when present.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own box-labelled images as a `.zip` holding `boxes.csv` (columns `id`, `file`, `x_min`, `y_min`, `x_max`, `y_max`, optional `group`; one row per box, pixel coordinates) beside the image files — images are decoded from the archive, never extracted to disk. They pass through the same validation, seeded group-disjoint split, prior, zero-shot scoring, frozen-policy heads, unfrozen-policy training and selection, held-out evaluation, detection rendering, artifact export and reload-parity cells as the Open Food Facts sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

At inference the DETR-style model reads one image resized to 800 px on its shortest edge, runs a ResNet-18 backbone and a 6-layer encoder–decoder, and emits exactly 15 query proposals, each a box and a softmax over `table`, `table rotated` and *no object*; the processor keeps the queries whose class score reaches the threshold and maps their boxes back to input pixels. The carried pipeline module adds snapshot verification, the input contract, a fixed output contract and the `box_iou`, `validate_inputs` and `evaluation_report` helpers; `evaluation_report` becomes `sample-sanity` only when a caller supplies reference boxes — which this notebook, unlike its inference-only predecessor, does, on a real photograph.

What this notebook adds to inference is **supervised adaptation to a new box class on a new image domain, under an explicit frozen-vs-unfrozen policy**. The dataset is real and far from PubTables-1M's PDF renders: 121 product photographs from the Open Food Facts nutrition-table detection set (version 1.1, manually reviewed; images CC BY-SA 3.0), each with one to three boxes around the printed nutrition table, pinned per file by byte size and SHA-256 of the served original, fetched at run time, refused on any mismatch and downscaled to 1,280 px. Two products have two photographs, so the sample is split by **product**, never by photograph. The carried `metrics.py` scores a prediction set by **AP@0.5**, **AP@0.75** and **mAP** (the COCO convention, one class, over every query of every image) plus the recall and precision at the pipeline's operating threshold and at 0.5; a **fixed-box prior** (the training split's mean box) and the **untouched checkpoint** (its `table` + `table rotated` mass as the score) frame the numbers. The **frozen policy** trains a new two-way class head and a copy of the box head on the frozen decoder features under the DETR set loss (Hungarian matching, cross-entropy with a 0.1 no-object weight, L1 and GIoU — implemented in the carried module, no external matcher); the **unfrozen policy** continues by training the last decoder layers with them end to end, and the epoch with the lowest validation loss — which may be the heads alone — is kept. The PubTables heads and `detect` are never trained or exported. The adaptation question is whether unfreezing the decoder buys anything over new heads on 72 photographs. Nothing here is a quality claim about your images: it is one seeded split of one small corpus.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, metrics and dataset modules guarantee; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned real box-labelled corpus and validate and split it by product without leakage; run a real photograph through the public detection API and read a `sample-sanity` report against reference boxes; read AP@0.5 / AP@0.75 / mAP beside a fixed-box prior and the zero-shot checkpoint and understand why a detector's score threshold is an operating point, not part of AP; train new detection heads on frozen features and a bounded decoder unfreeze with explicit hyperparameters and validation-based selection between the two policies; evaluate on an independent product-disjoint test split; compare detections before and after; and export a safetensors adapter (heads plus any trained decoder layers) that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** table *structure* recognition (rows, columns, cells — the sibling `table-transformer-structure-pipeline` covers it), OCR or nutrient text extraction, multi-class detection, backbone or encoder training, data augmentation, any training of the PubTables heads, any PubTables-1M accuracy claim, and any claim that 121 product photographs from one site stand in for your images. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; float32 on both. The build record measured about 0.3 s per photograph to run the decoder on CPU (3 s for the 25-photograph zero-shot pass), about 20 s for the new heads including feature extraction, and about 30 s per unfreeze epoch over 72 photographs plus a 24-photograph validation pass. The pinned `torch==2.14.0` install and the 115 MB checkpoint are the large downloads of the run, then the 129 MB of photographs.
- **Knowledge:** basic Python and PIL; what a bounding box in xyxy pixel coordinates is; what intersection-over-union and average precision measure and why AP does not depend on a score threshold; what a Hungarian (one-to-one) matching between predictions and references is; what validation-based selection between two policies means.
- **Data contract:** records are `{{id, image, boxes}}` — a PIL image (or a path to one) with sides 16..4,096 px, a list of 1..15 `[x_min, y_min, x_max, y_max]` pixel boxes inside the image with sides of at least 4 px, ids matching `[A-Za-z0-9_.:-]{{1,64}}` and unique; a training set needs 8..2,000 records; images are de-duplicated by decoded-pixel digest and split by `group` / `barcode` so one product never straddles splits. BYOD accepts a `.zip` (or a directory) holding `boxes.csv` and the image files.
- **Validation is structural, not semantic:** nothing checks that a box is around a table — a mislabelled set is trained on without complaint; all boxes are one class, and the four Open Food Facts nutrition-table categories are merged into it (kept under `categories` for provenance).
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — product photographs can show hands, homes and receipts. The default path uploads nothing.
- **External access (data):** besides the Hub, the default path fetches 121 pinned objects (`<barcode path>/<image id>.jpg`, 129,318,620 bytes in total, one SHA-256 each in the carried `SAMPLE_RECORDS` table) from `static.openfoodfacts.org` over HTTPS, each refused on any byte-size or SHA-256 mismatch before it is decoded; the images are CC BY-SA 3.0 (Open Food Facts contributors) and the boxes come from the Open Food Facts nutrition-table detection dataset (ODbL), credited in the References.
- **External access:** the Hugging Face Hub only, to fetch the pinned `microsoft/table-transformer-detection` snapshot (~115 MB in total) at revision `2357cbe2b5a5…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `timm` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'timm==1.0.29',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'table-transformer-detection-pipeline',
    'repository_revision': '1530d418406cf379318e7d8a47f0bc77bbae6d2a',
    'embedded_module': 'src/table_transformer_detection_pipeline/pipeline.py',
    'embedded_modules': ['src/table_transformer_detection_pipeline/pipeline.py', 'src/table_transformer_detection_pipeline/metrics.py', 'src/table_transformer_detection_pipeline/samples.py'],
    'module_sha256': '87e6327197e4d2223d523c9181e81ef8ab79d11bcd1a465b8661f8513bacb30b',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, timm
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'timm': timm.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/table_transformer_detection_pipeline/` @ `1530d418406c`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/table_transformer_detection_pipeline/pipeline.py`

In [ ]:
from __future__ import annotations

import copy
import hashlib
import itertools
import json
import math
import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "microsoft/table-transformer-detection"
MODEL_REVISION = "2357cbe2b5a5d1c03e54f32764f06058933b65ab"
MODEL_LICENSE = "mit"
MODEL_KEY = "table-transformer-detection"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# The two classes the checkpoint was fine-tuned on (config.json id2label). A "table rotated" box is a
# table whose text runs vertically; the pipeline returns the label as-is and does not rotate anything.
LABELS = ("table", "table rotated")
# Detection threshold: the value the Transformers Table Transformer documentation example passes to
# post_process_object_detection (threshold=0.9). It gates a softmax class score over 15 DETR queries
# that was not calibrated for any document domain; the deployment owns tuning it on labelled pages.
DETECTION_THRESHOLD = 0.9
# The checkpoint's DETR decoder emits exactly num_queries proposals per page (config.json), so no
# page can yield more than this many boxes.
MAX_DETECTIONS = 15
# Input ceilings. The processor resizes the shortest edge to 800 px with the longest capped at 800
# (preprocessor_config.json `size`/`max_size`), so image cost is bounded whatever the caller sends;
# the side ceiling only guards memory during decoding and resizing.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
WEIGHTS_FILE = "model.safetensors"
WEIGHT_SHA256 = (
    "8f1aa73170102c038d40155e2734b343bf07e0fe12594228a8590943b01dccf7"  # manifest digest of WEIGHTS_FILE
)
PARAMETER_COUNT = 28_799_431  # ResNet-18 backbone 11,166,912 + encoder 7,890,944 + decoder 9,473,024 + heads
D_MODEL = 256
NUM_QUERIES = MAX_DETECTIONS
DECODER_LAYERS = 6
DEFAULT_TRAINABLE_LAYERS = (
    2  # the unfrozen policy trains the last two decoder layers (3,157,504 parameters) with the heads
)
MAX_EVAL_RECORDS = 2_000
MIN_SCORED_RECORDS = 50  # below this a scored dataset is labelled a small sample
CLASS_COST, BBOX_COST, GIOU_COST = 1.0, 5.0, 2.0  # DETR matching costs and loss weights (upstream defaults)
NO_OBJECT_WEIGHT = 0.1  # DETR eos_coef
EXACT_MATCH_MAX_TARGETS = 4  # up to this many reference boxes the Hungarian assignment is enumerated exactly
ARTIFACT_FORMAT = "org.valcorza.table-transformer-detection.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
POLICY_FROZEN = "frozen backbone, encoder and decoder + new heads"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    """Intersection-over-union of two xyxy pixel boxes; the building block for any caller-side mAP."""
    if len(a) != 4 or len(b) != 4:
        raise ValueError("boxes must be [x0, y0, x1, y1]")
    if a[2] < a[0] or a[3] < a[1] or b[2] < b[0] or b[3] < b[1]:
        raise ValueError("boxes must satisfy x0 <= x1 and y0 <= y1")
    inter_w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return float(inter / union) if union > 0 else 0.0


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(value: Any) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"threshold must be a number in [0, 1], got {value!r}")
    return float(value)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one page image as PIL.Image.Image (any mode, converted to RGB): a rendered page or a scan",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "threshold": [0.0, 1.0],
    "labels": list(LABELS),
    "max_detections": MAX_DETECTIONS,
    "preprocessing": (
        "image converted to RGB; the processor resizes to shortest edge 800 px (longest edge capped at "
        "800), normalises with ImageNet mean/std, and returned boxes are mapped back to input pixels"
    ),
}


def _check_inputs(image: Any, threshold: Any) -> tuple[Image.Image, float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``detect`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    return validate_image(image), _check_threshold(threshold)


def validate_inputs(
    image: Image.Image,
    *,
    threshold: float = DETECTION_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``detect`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, checked = _check_inputs(image, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (detect takes one page image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "threshold": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    ground_truth_boxes: Sequence[Sequence[float]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``ground_truth_boxes`` (xyxy reference boxes of the tables on the page) the report carries
    one ``box_iou`` entry per reference — the best-overlapping detection — as sample-sanity geometry
    evidence; without them the verdict is ``not-measurable`` and the report says what labelled data
    would make the task measurable.
    """
    detections = list(result["detections"])
    base = {
        "task": "table detection on document page images",
        "decision_rule": (
            "a DETR query survives when its softmax score for `table` or `table rotated` reaches the "
            "threshold; the score is a class probability under the model's own softmax, not a calibrated "
            "estimate for the deployment's pages"
        ),
        "threshold": result.get("threshold", DETECTION_THRESHOLD),
        "sample_kind": sample_kind,
        "n_detections": len(detections),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not ground_truth_boxes:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth table boxes were supplied for the evaluated page",
            "needs": (
                "labelled table boxes on your own pages, scored per table with box_iou and aggregated into "
                "precision/recall or mean average precision at a stated IoU threshold; no such labelled set "
                "ships with this repository"
            ),
        }
    metrics = []
    for index, box in enumerate(ground_truth_boxes):
        ious = [box_iou(det["box"], box) for det in detections]
        best = max(range(len(ious)), key=ious.__getitem__) if ious else None
        metrics.append(
            {
                "id": "box_iou",
                "reference": f"table-{index}",
                "value": ious[best] if best is not None else 0.0,
                "matched_label": detections[best]["label"] if best is not None else None,
                "estimation": "one reference box per table on a single page, no dispersion estimate",
            }
        )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} reference box(es) on one tutorial page; geometry sanity evidence, "
            "not a detection benchmark"
        ),
        "needs": (
            "a labelled page set from the deployment domain (scans, renders, layouts) for any "
            "mean-average-precision or precision/recall claim"
        ),
    }


@dataclass
class TableTransformerDetectionPipeline:
    """Table detection on document page images over the pinned Table Transformer (DETR-R18) checkpoint.

    The adaptation contract (`evaluate_zero_shot`, `adapt`, `evaluate`, `detect_adapted`,
    `save_artifact`, `from_artifact`) trains a **new** class head and a copy of the box head over the
    frozen DETR decoder features of a validated `{id, image, boxes}` dataset (the **frozen policy**),
    optionally continues with a bounded unfreeze of the last decoder layers (the **unfrozen policy**),
    scores held-out images by AP@0.5 / AP@0.75 / mAP against a fixed-box prior and the untouched
    checkpoint (the zero-shot row), and exports the trained tensors as a safetensors adapter bound to
    the pinned base weights. `detect` and the PubTables heads are unchanged by it, but `detect` reads
    the adapted decoder once an unfreeze has run."""

    _runner: Callable[[Image.Image, float], list[dict[str, Any]]]
    device: str
    source: str = "injected"
    _model: Any = field(default=None, repr=False)
    _processor: Any = field(default=None, repr=False)
    _head: Any = field(default=None, repr=False)
    _bbox_head: Any = field(default=None, repr=False)
    classes: list[str] | None = field(default=None, repr=False)
    adapter: dict[str, Any] | None = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> TableTransformerDetectionPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs, origin = str(root), {"local_files_only": True}, "local-snapshot"
        elif allow_download:
            source, kwargs, origin = MODEL_ID, {}, "hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoImageProcessor, TableTransformerForObjectDetection

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = AutoImageProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        # config.json says use_pretrained_backbone=true, which would make the timm ResNet-18 backbone
        # fetch ImageNet weights from the Hub at construction time — an unpinned download that the
        # checkpoint immediately overwrites. The backbone weights are in model.safetensors; refuse it.
        model = TableTransformerForObjectDetection.from_pretrained(
            source,
            revision=MODEL_REVISION,
            trust_remote_code=False,
            use_pretrained_backbone=False,
            **kwargs,
        )
        model = model.to(resolved_device).eval()
        id2label = {int(k): v for k, v in model.config.id2label.items()}

        def runner(image: Image.Image, threshold: float) -> list[dict]:
            inputs = processor(images=image, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                outputs = model(**inputs)
            result = processor.post_process_object_detection(
                outputs, threshold=threshold, target_sizes=[image.size[::-1]]
            )[0]
            return [
                {
                    "box": [float(v) for v in box.tolist()],
                    "label": id2label[int(label)],
                    "score": float(score),
                }
                for box, label, score in zip(result["boxes"], result["labels"], result["scores"], strict=True)
            ]

        return cls(runner, resolved_device, origin, _model=model, _processor=processor)

    def detect(self, image: Image.Image, *, threshold: float = DETECTION_THRESHOLD) -> dict[str, Any]:
        """Detect tables on one page image; boxes are xyxy pixel coordinates in the input image."""
        rgb, checked = _check_inputs(image, threshold)
        detections = self._runner(rgb, checked)
        if len(detections) > MAX_DETECTIONS:
            raise RuntimeError(
                f"backend returned {len(detections)} detections > num_queries {MAX_DETECTIONS}"
            )
        for det in detections:
            if set(det) != {"box", "label", "score"} or len(det["box"]) != 4 or det["label"] not in LABELS:
                raise RuntimeError(f"backend returned a malformed detection: {det!r}")
        return {
            "detections": sorted(detections, key=lambda d: -d["score"]),
            "threshold": checked,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation ----
    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._processor

    def _require_heads(self) -> tuple[Any, Any, list[str]]:
        if self._head is None or self._bbox_head is None or not self.classes:
            raise ValueError("no adapted heads: call adapt() or load an artifact first")
        return self._head, self._bbox_head, list(self.classes)

    def _trainable_names(self, trainable_layers: int) -> list[str]:
        if isinstance(trainable_layers, bool) or not isinstance(trainable_layers, int):
            raise ValueError(f"trainable_layers must be an int in 0..{DECODER_LAYERS}")
        if not 0 <= trainable_layers <= DECODER_LAYERS:
            raise ValueError(f"trainable_layers must be an int in 0..{DECODER_LAYERS}")
        if trainable_layers == 0:
            return []
        model, _ = self._require_model()
        first = DECODER_LAYERS - trainable_layers
        prefixes = tuple(f"model.decoder.layers.{k}." for k in range(first, DECODER_LAYERS))
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def _decoder_features(self, image: Image.Image, *, grad: bool = False) -> Any:
        """The (NUM_QUERIES, D_MODEL) decoder output for one validated image through the processor."""
        import torch

        model, processor = self._require_model()
        inputs = processor(images=image, return_tensors="pt").to(self.device)
        if grad:
            outputs = model.model(pixel_values=inputs["pixel_values"], pixel_mask=inputs.get("pixel_mask"))
        else:
            with torch.no_grad():
                outputs = model.model(
                    pixel_values=inputs["pixel_values"], pixel_mask=inputs.get("pixel_mask")
                )
        hidden = outputs.last_hidden_state[0]
        if tuple(hidden.shape) != (NUM_QUERIES, D_MODEL):
            raise RuntimeError(f"decoder returned {tuple(hidden.shape)}, expected {(NUM_QUERIES, D_MODEL)}")
        return hidden

    @staticmethod
    def _xyxy_pixels(boxes_cxcywh: Any, width: int, height: int) -> list[list[float]]:
        out = []
        for cx, cy, w, h in boxes_cxcywh.tolist():
            out.append(
                [(cx - w / 2) * width, (cy - h / 2) * height, (cx + w / 2) * width, (cy + h / 2) * height]
            )
        return out

    def evaluate_zero_shot(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Score the untouched checkpoint on validated `{id, image, boxes}` records with its PubTables
        heads: a query's score is its softmax mass on `table` + `table rotated` (the closest thing
        the checkpoint has to the adapted class); AP over all 15 queries per image, the operating
        point at DETECTION_THRESHOLD.
        """
        import torch

        pass  # standalone rewrite (build_notebook.py): `from .metrics import detection_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        model, _ = self._require_model()
        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        predictions = []
        for record in checked:
            hidden = self._decoder_features(record["image"])
            with torch.no_grad():
                probabilities = torch.softmax(model.class_labels_classifier(hidden), dim=-1)
                boxes = model.bbox_predictor(hidden).sigmoid()
            scores = probabilities[:, : len(LABELS)].sum(dim=-1)
            xyxy = self._xyxy_pixels(boxes, *record["image"].size)
            predictions.append(
                [{"box": b, "score": float(s)} for b, s in zip(xyxy, scores.tolist(), strict=True)]
            )
        metrics = detection_metrics(predictions, [r["boxes"] for r in checked], threshold=DETECTION_THRESHOLD)
        metrics["policy"] = "zero-shot checkpoint (PubTables heads, table + table rotated mass as the score)"
        metrics["adapted"] = False
        metrics["verdict"] = "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample"
        metrics["seconds"] = round(time.perf_counter() - started, 3)
        metrics["model_id"] = MODEL_ID
        metrics["model_revision"] = MODEL_REVISION
        return metrics

    def predict_boxes(self, records: Sequence[Mapping[str, Any]]) -> list[list[dict[str, Any]]]:
        """Every query's adapted-class score and pixel box, per validated record (the ranking AP consumes)."""
        import torch

        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        head, bbox_head, _classes = self._require_heads()
        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        out = []
        for record in checked:
            hidden = self._decoder_features(record["image"])
            with torch.no_grad():
                probabilities = torch.softmax(head(hidden), dim=-1)
                boxes = bbox_head(hidden).sigmoid()
            xyxy = self._xyxy_pixels(boxes, *record["image"].size)
            out.append(
                [
                    {"box": b, "score": float(s)}
                    for b, s in zip(xyxy, probabilities[:, 0].tolist(), strict=True)
                ]
            )
        return out

    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Score the adapted heads on validated labelled images: AP@0.5 / AP@0.75 / mAP plus the operating
        point at DETECTION_THRESHOLD, and the DETR loss the selection used."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import detection_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        predictions = self.predict_boxes(checked)
        metrics = detection_metrics(predictions, [r["boxes"] for r in checked], threshold=DETECTION_THRESHOLD)
        metrics["loss"] = self._dataset_loss(checked)
        metrics["policy"] = self.adapter["policy"] if self.adapter else "unknown"
        metrics["adapted"] = True
        metrics["verdict"] = "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample"
        metrics["seconds"] = round(time.perf_counter() - started, 3)
        metrics["model_id"] = MODEL_ID
        metrics["model_revision"] = MODEL_REVISION
        return metrics

    def detect_adapted(self, image: Image.Image, *, threshold: float = DETECTION_THRESHOLD) -> dict[str, Any]:
        """`detect` with the adapted heads: boxes of the adapted class at or above `threshold`, in pixels."""
        rgb, checked = _check_inputs(image, threshold)
        head, bbox_head, classes = self._require_heads()
        queries = self.predict_boxes(
            [{"id": "query", "image": rgb, "boxes": [[0, 0, rgb.width, rgb.height]]}]
        )[0]
        detections = [
            {"box": q["box"], "label": classes[0], "score": q["score"]}
            for q in queries
            if q["score"] >= checked
        ]
        return {
            "detections": sorted(detections, key=lambda d: -d["score"]),
            "threshold": checked,
            "classes": classes,
            "policy": self.adapter["policy"] if self.adapter else "unknown",
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- the DETR set loss, implemented here (no scipy) ----
    @staticmethod
    def _targets(record: Mapping[str, Any]) -> Any:
        import torch

        width, height = record["image"].size
        rows = []
        for x0, y0, x1, y1 in record["boxes"]:
            rows.append(
                [(x0 + x1) / 2 / width, (y0 + y1) / 2 / height, (x1 - x0) / width, (y1 - y0) / height]
            )
        return torch.tensor(rows, dtype=torch.float32)

    @staticmethod
    def _giou(a: Any, b: Any) -> Any:
        """Generalised IoU between (N, 4) and (M, 4) cxcywh boxes -> (N, M)."""
        import torch

        def xyxy(box):
            return torch.stack(
                [
                    box[:, 0] - box[:, 2] / 2,
                    box[:, 1] - box[:, 3] / 2,
                    box[:, 0] + box[:, 2] / 2,
                    box[:, 1] + box[:, 3] / 2,
                ],
                dim=-1,
            )

        a, b = xyxy(a), xyxy(b)
        area_a = (a[:, 2] - a[:, 0]).clamp_min(0) * (a[:, 3] - a[:, 1]).clamp_min(0)
        area_b = (b[:, 2] - b[:, 0]).clamp_min(0) * (b[:, 3] - b[:, 1]).clamp_min(0)
        lt = torch.max(a[:, None, :2], b[None, :, :2])
        rb = torch.min(a[:, None, 2:], b[None, :, 2:])
        inter = (rb - lt).clamp_min(0).prod(dim=-1)
        union = area_a[:, None] + area_b[None, :] - inter
        iou = inter / union.clamp_min(1e-9)
        lt_c = torch.min(a[:, None, :2], b[None, :, :2])
        rb_c = torch.max(a[:, None, 2:], b[None, :, 2:])
        enclosing = (rb_c - lt_c).clamp_min(0).prod(dim=-1)
        return iou - (enclosing - union) / enclosing.clamp_min(1e-9)

    def _match(self, probabilities: Any, boxes: Any, targets: Any) -> list[tuple[int, int]]:
        """Minimum-cost one-to-one assignment of reference boxes to queries (exact for small sets)."""
        import torch

        with torch.no_grad():
            cost = -CLASS_COST * probabilities[:, :1].expand(-1, targets.shape[0])
            cost = (
                cost + BBOX_COST * torch.cdist(boxes, targets, p=1) - GIOU_COST * self._giou(boxes, targets)
            )
            cost = cost.cpu().numpy()
        n_targets = targets.shape[0]
        if n_targets <= EXACT_MATCH_MAX_TARGETS:
            best, best_perm = math.inf, None
            for perm in itertools.permutations(range(NUM_QUERIES), n_targets):
                total = sum(cost[q, g] for g, q in enumerate(perm))
                if total < best:
                    best, best_perm = total, perm
            return [(q, g) for g, q in enumerate(best_perm)]
        pairs = []
        used: set[int] = set()
        order = sorted((cost[q, g], q, g) for q in range(NUM_QUERIES) for g in range(n_targets))
        assigned: set[int] = set()
        for _c, q, g in order:
            if q in used or g in assigned:
                continue
            pairs.append((q, g))
            used.add(q)
            assigned.add(g)
        return pairs

    def _loss(self, logits: Any, boxes: Any, targets: Any) -> Any:
        """DETR set loss for one image: weighted cross-entropy over queries (no-object weight 0.1), L1 and
        GIoU on the matched boxes, normalised by the number of reference boxes."""
        import torch

        probabilities = torch.softmax(logits, dim=-1)
        pairs = self._match(probabilities.detach(), boxes.detach(), targets)
        n_classes = logits.shape[-1] - 1
        target_classes = torch.full((NUM_QUERIES,), n_classes, dtype=torch.long, device=logits.device)
        query_index = torch.tensor([q for q, _g in pairs], dtype=torch.long, device=logits.device)
        target_index = torch.tensor([g for _q, g in pairs], dtype=torch.long, device=logits.device)
        target_classes[query_index] = 0
        weights = torch.ones(n_classes + 1, device=logits.device)
        weights[n_classes] = NO_OBJECT_WEIGHT
        loss_ce = torch.nn.functional.cross_entropy(logits, target_classes, weight=weights)
        matched_boxes = boxes[query_index]
        matched_targets = targets.to(logits.device)[target_index]
        loss_bbox = torch.nn.functional.l1_loss(matched_boxes, matched_targets, reduction="sum") / max(
            len(pairs), 1
        )
        loss_giou = (1.0 - torch.diagonal(self._giou(matched_boxes, matched_targets))).sum() / max(
            len(pairs), 1
        )
        return loss_ce + BBOX_COST * loss_bbox + GIOU_COST * loss_giou

    def _dataset_loss(self, checked: Sequence[Mapping[str, Any]]) -> float:
        import torch

        head, bbox_head, _classes = self._require_heads()
        total = 0.0
        for record in checked:
            hidden = self._decoder_features(record["image"])
            with torch.no_grad():
                total += float(self._loss(head(hidden), bbox_head(hidden).sigmoid(), self._targets(record)))
        return total / max(len(checked), 1)

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        head_steps: int = 300,
        head_lr: float = 1e-3,
        trainable_layers: int = DEFAULT_TRAINABLE_LAYERS,
        epochs: int = 3,
        lr: float = 1e-4,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Two-stage bounded adaptation, selected on validation.

        Stage A (the **frozen policy**): a new `Linear(D_MODEL, 2)` class head (`nutrition-table` /
        no-object) and a copy of the checkpoint's box head trained on the cached decoder features of
        the training images with AdamW (`head_lr`, weight decay 1e-4) for `head_steps` full-batch
        steps under the DETR set loss — recorded as epoch 0. Stage B (the **unfrozen policy**, when
        `trainable_layers` > 0 and `epochs` > 0): the last `trainable_layers` decoder layers trained
        with both heads end to end, one image per step, for `epochs` epochs (AdamW at `lr`, weight
        decay 0.01, gradient clipping 0.1, seeded order, no augmentation); the backbone, the input
        projection, the encoder, the query embeddings and the earlier decoder layers stay frozen.
        Every epoch is scored on `val` and the epoch with the **lowest validation DETR loss** is
        kept (epoch 0 competes) and its tensors restored; without `val` the final epoch is kept.
        """
        if isinstance(head_steps, bool) or not isinstance(head_steps, int) or not 1 <= head_steps <= 5_000:
            raise ValueError("head_steps must be an int in 1..5000")
        if not isinstance(head_lr, int | float) or not 0.0 < float(head_lr) <= 1e-1:
            raise ValueError("head_lr must be in (0, 0.1]")
        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 0 <= epochs <= 20:
            raise ValueError("epochs must be an int in 0..20")
        if not isinstance(lr, int | float) or not 0.0 < float(lr) <= 1e-2:
            raise ValueError("lr must be in (0, 1e-2]")
        names = self._trainable_names(trainable_layers)
        model, _ = self._require_model()
        import torch

        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        train_records = validate_dataset(train)["records"]
        val_records = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
            if val is not None
            else None
        )
        classes = [self.classes[0] if self.classes else "nutrition-table"]
        started = time.perf_counter()
        torch.manual_seed(seed)
        rng = random.Random(seed)
        model.eval()
        for p in model.parameters():
            p.requires_grad_(False)

        # Stage A: the heads on cached decoder features.
        cached = [self._decoder_features(r["image"]) for r in train_records]
        targets = [self._targets(r) for r in train_records]
        head = torch.nn.Linear(D_MODEL, len(classes) + 1).to(self.device)
        bbox_head = copy.deepcopy(model.bbox_predictor).to(self.device)
        for p in bbox_head.parameters():
            p.requires_grad_(True)
        head_opt = torch.optim.AdamW(
            [*head.parameters(), *bbox_head.parameters()], lr=float(head_lr), weight_decay=1e-4
        )
        head_loss = math.nan
        for _ in range(head_steps):
            head_opt.zero_grad(set_to_none=True)
            loss = sum(
                self._loss(head(h), bbox_head(h).sigmoid(), tg) for h, tg in zip(cached, targets, strict=True)
            ) / len(cached)
            loss.backward()
            head_opt.step()
            head_loss = float(loss.detach())
        self._head, self._bbox_head, self.classes = head, bbox_head, classes
        self.adapter = {"policy": POLICY_FROZEN}

        def brief(metrics: Mapping[str, Any] | None) -> dict[str, float] | None:
            if metrics is None:
                return None
            return {
                "n": metrics["n_images"],
                "loss": round(metrics["loss"], 6),
                "ap50": round(metrics["ap50"], 6),
                "map": round(metrics["map"], 6),
                "recall_at_threshold": round(metrics["recall_at_threshold"], 6),
            }

        def score() -> dict[str, float] | None:
            head.eval()
            bbox_head.eval()
            model.eval()
            return brief(self.evaluate(val_records)) if val_records is not None else None

        history: list[dict[str, Any]] = [
            {
                "epoch": 0,
                "stage": "new heads on frozen decoder features",
                "train_loss": head_loss,
                "val": score(),
            }
        ]
        params = dict(model.named_parameters())
        best_epoch = 0
        best_score = history[0]["val"]["loss"] if history[0]["val"] else math.inf
        best_state = {
            "head": {k: v.detach().clone() for k, v in head.state_dict().items()},
            "bbox_head": {k: v.detach().clone() for k, v in bbox_head.state_dict().items()},
            "layers": {n: params[n].detach().clone() for n in names},
        }
        policy = POLICY_FROZEN
        if names and epochs > 0:
            policy_b = f"unfrozen last {trainable_layers} decoder layers + new heads"
            for n in names:
                params[n].requires_grad_(True)
            optimiser = torch.optim.AdamW(
                [*head.parameters(), *bbox_head.parameters(), *(params[n] for n in names)],
                lr=float(lr),
                weight_decay=0.01,
            )
            for epoch in range(1, epochs + 1):
                model.train()
                head.train()
                bbox_head.train()
                order = list(range(len(train_records)))
                rng.shuffle(order)
                losses = []
                for index in order:
                    record = train_records[index]
                    hidden = self._decoder_features(record["image"], grad=True)
                    loss = self._loss(head(hidden), bbox_head(hidden).sigmoid(), targets[index])
                    optimiser.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(
                        [*head.parameters(), *bbox_head.parameters(), *(params[n] for n in names)], 0.1
                    )
                    optimiser.step()
                    losses.append(float(loss.detach()))
                self.adapter = {"policy": policy_b}
                val_metrics = score()
                entry = {
                    "epoch": epoch,
                    "stage": policy_b,
                    "train_loss": float(sum(losses) / len(losses)),
                    "val": val_metrics,
                }
                history.append(entry)
                if progress is not None:
                    progress(entry)
                current = val_metrics["loss"] if val_metrics else -epoch  # no val: the last epoch wins
                if current < best_score:
                    best_epoch, best_score, policy = epoch, current, policy_b
                    best_state = {
                        "head": {k: v.detach().clone() for k, v in head.state_dict().items()},
                        "bbox_head": {k: v.detach().clone() for k, v in bbox_head.state_dict().items()},
                        "layers": {n: params[n].detach().clone() for n in names},
                    }
            with torch.no_grad():
                head.load_state_dict(best_state["head"])
                bbox_head.load_state_dict(best_state["bbox_head"])
                for n, value in best_state["layers"].items():
                    params[n].copy_(value)
        for p in model.parameters():
            p.requires_grad_(False)
        for p in [*head.parameters(), *bbox_head.parameters()]:
            p.requires_grad_(False)
        model.eval()
        head.eval()
        bbox_head.eval()
        self.adapter = {
            "policy": policy,
            "classes": classes,
            "head_steps": head_steps,
            "head_lr": float(head_lr),
            "head_final_loss": head_loss,
            "trainable_layers": trainable_layers,
            "n_trainable_head": sum(p.numel() for p in head.parameters())
            + sum(p.numel() for p in bbox_head.parameters()),
            "n_trainable_layers": sum(params[n].numel() for n in names),
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "lowest validation DETR loss (epoch 0 = new heads on frozen features)"
            if val is not None
            else "final epoch (no validation split)",
            "lr": float(lr),
            "n_train": len(train_records),
            "n_val": len(val_records) if val_records is not None else 0,
            "seed": seed,
            "history": history,
            "trainable_names": names if policy != POLICY_FROZEN else [],
            "seconds": round(time.perf_counter() - started, 3),
        }
        return dict(self.adapter)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the new heads (and any trained decoder-layer tensors) as safetensors with a base manifest."""
        if self.adapter is None or self._head is None or self._bbox_head is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter.get("trainable_names", []))
        tensors = {f"head.{k}": v.detach().cpu().contiguous() for k, v in self._head.state_dict().items()}
        tensors.update(
            {f"bbox_head.{k}": v.detach().cpu().contiguous() for k, v in self._bbox_head.state_dict().items()}
        )
        tensors.update(
            {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        )
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHTS_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter.get("history", []),
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest and digest **before** deserialising, rebuild the heads and overlay any
        decoder-layer tensors onto the base."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        entry = manifest["files"][0]
        weights_path = root / entry["path"]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        classes = list(manifest.get("adapter", {}).get("classes") or [])
        if len(classes) < 1:
            raise ValueError("artifact manifest does not name the adapted class")
        model, _ = self._require_model()
        import torch
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != manifest["tensors"]:
            raise ValueError("artifact tensor names differ from its manifest")
        if (
            tuple(tensors.get("head.weight", torch.empty(0)).shape) != (len(classes) + 1, D_MODEL)
            or "head.bias" not in tensors
        ):
            raise ValueError("artifact class head does not match D_MODEL and the manifest's classes")
        bbox_head = copy.deepcopy(model.bbox_predictor)
        bbox_state = {k[len("bbox_head.") :]: v for k, v in tensors.items() if k.startswith("bbox_head.")}
        if set(bbox_state) != set(bbox_head.state_dict()):
            raise ValueError("artifact box head tensors do not match the checkpoint's box head")
        state = model.state_dict()
        layer_tensors = {k: v for k, v in tensors.items() if not k.startswith(("head.", "bbox_head."))}
        for key, value in layer_tensors.items():
            if key not in state or not key.startswith("model.decoder.layers."):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable decoder-layer tensor of the base"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key}: shape {tuple(value.shape)} != {tuple(state[key].shape)}"
                )
        head = torch.nn.Linear(D_MODEL, len(classes) + 1)
        head.load_state_dict({"weight": tensors["head.weight"].float(), "bias": tensors["head.bias"].float()})
        bbox_head.load_state_dict({k: v.float() for k, v in bbox_state.items()})
        head = head.to(self.device).eval()
        bbox_head = bbox_head.to(self.device).eval()
        for p in [*head.parameters(), *bbox_head.parameters()]:
            p.requires_grad_(False)
        if layer_tensors:
            with torch.no_grad():
                params = dict(model.named_parameters())
                for key, value in layer_tensors.items():
                    params[key].copy_(value.to(params[key].dtype))
            model.eval()
        self._head, self._bbox_head, self.classes = head, bbox_head, classes
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": sorted(layer_tensors),
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> TableTransformerDetectionPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 2/3:** `src/table_transformer_detection_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Detection metrics for the adaptation contract, implemented here with no external scorer.

Boxes are ``[x_min, y_min, x_max, y_max]`` in pixels of the image they belong to. A prediction is a list of
``{"box", "score"}`` entries per image (every DETR query, not just those above a threshold — average
precision ranks them all); the reference is the list of ground-truth boxes per image.

- **AP@t** — average precision at IoU threshold *t*: predictions of all images are ranked by score, each is a
  true positive when it overlaps a not-yet-matched reference box of its image with IoU ≥ *t* (greedy in score
  order), the precision-recall curve is made monotone from the right and the area under it is summed over the
  recall steps (the VOC 2010+ / COCO "all-points" convention, one class).
- **mAP** — the mean of AP@t over t = 0.50, 0.55, …, 0.95 (the COCO primary metric, one class).
- **operating-point recall / precision** — at the pipeline's `DETECTION_THRESHOLD`, how many reference boxes
  are found (IoU ≥ 0.5) and how many surviving detections are true positives.
- **mean best IoU** — for every reference box, the IoU of its best-overlapping prediction, averaged.

The **fixed-box prior** frames the numbers: one box per image at the mean normalised reference box of the
training split, with a constant score — what "the table is usually here" alone buys.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any

import numpy as np

# standalone rewrite (build_notebook.py): `from .pipeline import box_iou` removed — names are kernel globals defined by the carried modules

IOU_THRESHOLDS = tuple(round(0.5 + 0.05 * i, 2) for i in range(10))
MATCH_IOU = 0.5

METRIC_DEFINITIONS = {
    "ap50": "average precision at IoU 0.50 over all ranked predictions of all images (one class)",
    "ap75": "average precision at IoU 0.75",
    "map": "mean of the average precision at IoU 0.50, 0.55, ..., 0.95 (the COCO primary metric, one class)",
    "recall_at_threshold": (
        "fraction of reference boxes matched (IoU >= 0.5) by a detection at or above the operating threshold"
    ),
    "precision_at_threshold": (
        "fraction of detections at or above the operating threshold that match a reference box"
    ),
    "mean_best_iou": "mean over reference boxes of the IoU of their best-overlapping prediction, any score",
    "operating_points": "recall / precision / surviving detections at the operating threshold and at 0.5",
}


def _check_pairs(
    predictions: Sequence[Sequence[Mapping[str, Any]]], references: Sequence[Sequence[Sequence[float]]]
) -> None:
    if len(predictions) != len(references):
        raise ValueError(f"{len(predictions)} prediction lists for {len(references)} reference lists")
    if not references:
        raise ValueError("at least one image is required")
    for preds in predictions:
        for p in preds:
            if "box" not in p or "score" not in p or len(p["box"]) != 4:
                raise ValueError("each prediction needs a 4-value 'box' and a 'score'")


def average_precision(
    predictions: Sequence[Sequence[Mapping[str, Any]]],
    references: Sequence[Sequence[Sequence[float]]],
    iou_threshold: float = MATCH_IOU,
) -> float:
    """All-points average precision for one class over a list of images."""
    _check_pairs(predictions, references)
    n_ref = sum(len(r) for r in references)
    if n_ref == 0:
        raise ValueError("at least one reference box is required")
    ranked = sorted(
        (
            (float(p["score"]), i, [float(v) for v in p["box"]])
            for i, preds in enumerate(predictions)
            for p in preds
        ),
        key=lambda t: -t[0],
    )
    matched = [np.zeros(len(r), dtype=bool) for r in references]
    tp = np.zeros(len(ranked))
    for k, (_score, i, box) in enumerate(ranked):
        best, best_j = 0.0, -1
        for j, ref in enumerate(references[i]):
            if matched[i][j]:
                continue
            iou = box_iou(box, ref)
            if iou > best:
                best, best_j = iou, j
        if best >= iou_threshold and best_j >= 0:
            matched[i][best_j] = True
            tp[k] = 1.0
    if not ranked:
        return 0.0
    cum_tp = np.cumsum(tp)
    cum_fp = np.cumsum(1.0 - tp)
    recall = cum_tp / n_ref
    precision = cum_tp / np.maximum(cum_tp + cum_fp, 1e-12)
    # monotone envelope from the right, area under the stepwise curve
    precision = np.maximum.accumulate(precision[::-1])[::-1]
    recall = np.concatenate([[0.0], recall])
    return float(np.sum((recall[1:] - recall[:-1]) * precision))


def _operating_point(
    predictions: Sequence[Sequence[Mapping[str, Any]]],
    references: Sequence[Sequence[Sequence[float]]],
    threshold: float,
) -> dict[str, Any]:
    """Recall / precision of the detections at or above `threshold` (greedy IoU >= MATCH_IOU matching)."""
    n_ref = sum(len(r) for r in references)
    found = kept = true_kept = 0
    per_image = []
    for preds, refs in zip(predictions, references, strict=True):
        surviving = [p for p in preds if float(p["score"]) >= threshold]
        kept += len(surviving)
        used: set[int] = set()
        image_found = 0
        for p in sorted(surviving, key=lambda q: -float(q["score"])):
            best, best_j = 0.0, -1
            for j, ref in enumerate(refs):
                if j in used:
                    continue
                iou = box_iou(p["box"], ref)
                if iou > best:
                    best, best_j = iou, j
            if best >= MATCH_IOU and best_j >= 0:
                used.add(best_j)
                image_found += 1
                true_kept += 1
        found += image_found
        per_image.append(
            {"n_reference": len(refs), "n_detections_at_threshold": len(surviving), "found": image_found}
        )
    return {
        "threshold": float(threshold),
        "recall": found / n_ref if n_ref else 0.0,
        "precision": true_kept / kept if kept else 0.0,
        "detections": kept,
        "per_image": per_image,
    }


def detection_metrics(
    predictions: Sequence[Sequence[Mapping[str, Any]]],
    references: Sequence[Sequence[Sequence[float]]],
    *,
    threshold: float,
) -> dict[str, Any]:
    """AP@0.5 / AP@0.75 / mAP over all ranked predictions, plus the operating point at `threshold`."""
    _check_pairs(predictions, references)
    aps = {t: average_precision(predictions, references, t) for t in IOU_THRESHOLDS}
    n_ref = sum(len(r) for r in references)
    points = {t_: _operating_point(predictions, references, t_) for t_ in sorted({0.5, float(threshold)})}
    primary = points[float(threshold)]
    best_ious = []
    per_image = []
    for preds, refs, image_point in zip(predictions, references, primary["per_image"], strict=True):
        image_best = [max((box_iou(p["box"], ref) for p in preds), default=0.0) for ref in refs]
        best_ious.extend(image_best)
        per_image.append({**image_point, "best_iou": [round(v, 4) for v in image_best]})
    return {
        "n_images": len(references),
        "n_reference_boxes": n_ref,
        "ap50": aps[0.5],
        "ap75": aps[0.75],
        "map": float(np.mean(list(aps.values()))),
        "ap_by_iou": {str(t): v for t, v in aps.items()},
        "threshold": float(threshold),
        "recall_at_threshold": primary["recall"],
        "precision_at_threshold": primary["precision"],
        "detections_at_threshold": primary["detections"],
        "operating_points": {
            str(t_): {k: v for k, v in point.items() if k != "per_image"} for t_, point in points.items()
        },
        "mean_best_iou": float(np.mean(best_ious)) if best_ious else 0.0,
        "per_image": per_image,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def fixed_box_prior(train_records: Sequence[Mapping[str, Any]]) -> list[float]:
    """The mean normalised reference box of the training split (x_min, y_min, x_max, y_max in 0..1)."""
    rows = []
    for record in train_records:
        w, h = record["image"].size
        for box in record["boxes"]:
            rows.append([box[0] / w, box[1] / h, box[2] / w, box[3] / h])
    if not rows:
        raise ValueError("the training split has no reference boxes")
    return [float(v) for v in np.mean(np.asarray(rows, dtype=np.float64), axis=0)]


def prior_baseline(
    train_records: Sequence[Mapping[str, Any]], test_records: Sequence[Mapping[str, Any]], *, threshold: float
) -> dict[str, Any]:
    """One box per test image at the training split's mean normalised box, scored 1.0."""
    prior = fixed_box_prior(train_records)
    predictions = []
    for record in test_records:
        w, h = record["image"].size
        predictions.append([{"box": [prior[0] * w, prior[1] * h, prior[2] * w, prior[3] * h], "score": 1.0}])
    out = detection_metrics(predictions, [r["boxes"] for r in test_records], threshold=threshold)
    out["baseline"] = (
        "one box per image at the training split's mean normalised reference box (fixed-box prior)"
    )
    out["prior_box_normalised"] = prior
    return out

**Module 3/3:** `src/table_transformer_detection_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Box-labelled image dataset contract for adapting the detector: the pinned Open Food Facts nutrition-table
sample, validation, seeded splitting, BYOD loaders and CSV export.

The default dataset is **real** and out of the checkpoint's document-page domain: 121 product photographs
from the Open Food Facts nutrition-table detection set (version 1.1, manually reviewed boxes; images CC BY-
SA 3.0, one to three nutrition-table boxes each), pinned here per file by byte size and SHA-256 of the
served original on `static.openfoodfacts.org` — the served originals were checked against the labelled
images (same size, no EXIF rotation, pixel MAE < 1/255), and the two whose served original carries an EXIF
orientation tag were left out of the 123-image split. The normalised boxes travel with each record. Every
file is fetched at run time and refused on any byte-size or SHA-256 mismatch, then downscaled to a longest
side of 1,280 px (the originals reach 5,312 px; the detector's processor resizes to 800 px anyway); the
repository redistributes none of the photographs, and each record keeps its barcode and image URL. Two
products have two photographs each, so the sample is split **by product**, never by photograph.

A record is ``{id, image, boxes}``: a PIL image (or a path to one) and a list of ``[x_min, y_min, x_max,
y_max]`` pixel boxes of the tables on it, all of one class (the four Open Food Facts nutrition-table
categories are merged into one `nutrition-table` class and kept under ``categories`` for provenance).
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_DETECTIONS, MAX_IMAGE_SIDE, MIN_IMAGE_SIDE, MODEL_ID` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "Open Food Facts nutrition-table photographs"
CORPUS_RELEASE = (
    "openfoodfacts/nutrition-table-detection v1.1 validation split (Hugging Face Hub commit d59dcee8), "
    "served originals pinned 2026-09-19"
)
CORPUS_BASE_URL = "https://static.openfoodfacts.org/images/products/"
CORPUS_LICENSE = "CC BY-SA 3.0 (Open Food Facts images; boxes from the Open Food Facts dataset, ODbL)"
CORPUS_BYTES = 129_318_620
CLASS_NAME = "nutrition-table"
CORPUS_LONGEST_SIDE = 1280  # served originals (up to 5,312 px) are downscaled to this at read time
SKIPPED_EXIF = ("0024138012322_3", "0041449003153_2")  # served originals with an EXIF orientation tag

# (record id, barcode, URL path under CORPUS_BASE_URL, width, height, bytes, sha256,
#  "ymin,xmin,ymax,xmax;..." normalised boxes, "category;..." per box)
SAMPLE_RECORDS: tuple[tuple[str, str, str, int, int, int, str, str, str], ...] = (
    (
        "0041129077641_1",
        "0041129077641",
        "004/112/907/7641/1.jpg",
        3120,
        4208,
        1998704,
        "d62e1721d6e1110fa129750a3718f0e8198abadc03801a0237ddbf986197e0d0",
        "0.487167,0.273397,0.640934,0.730449",
        "nutrition-table",
    ),
    (
        "0070200581159_1",
        "0070200581159",
        "007/020/058/1159/1.jpg",
        2448,
        3264,
        1657683,
        "adebe42a7b4d662ca5c1265eac6808ef308d66c58d5b6b3ceb4ce162b2b19e67",
        "0.246630,0.265523,0.453257,0.782555",
        "nutrition-table",
    ),
    (
        "0064144043064_2",
        "0064144043064",
        "006/414/404/3064/2.jpg",
        2000,
        2666,
        566828,
        "9608bb7e5ba84ef1efcc034cd01cbb517fb0d81b53ce9fc5c9fdf787588c3c52",
        "0.103151,0.040950,0.514332,0.938593",
        "nutrition-table",
    ),
    (
        "0051500700167_1",
        "0051500700167",
        "005/150/070/0167/1.jpg",
        2448,
        3264,
        719315,
        "78ada22c3c2f97f3808c47b591b91767d9b37c26d977fa0b04fc470fb62c20d5",
        "0.314339,0.164324,0.833649,0.584703",
        "nutrition-table",
    ),
    (
        "0016447100951_1",
        "0016447100951",
        "001/644/710/0951/1.jpg",
        2000,
        2697,
        460770,
        "1897c94d877f14f53c5388fc5060271a3ca2fb1d018a9cc4444a84cbfd3c68f1",
        "0.362996,0.350000,0.615499,0.559500",
        "nutrition-table",
    ),
    (
        "26195070_4",
        "26195070",
        "26195070/4.jpg",
        2000,
        1125,
        191335,
        "5555ecda7daf2ded79717034567ed6058aeb8461d1bbf4486cc1d4b64611e9e1",
        "0.144000,0.318864,0.807477,0.702636",
        "nutrition-table",
    ),
    (
        "0041244641024_3",
        "0041244641024",
        "004/124/464/1024/3.jpg",
        1675,
        1200,
        433717,
        "0d46658a1de8ed3f6facf5c2eb9ba1be56c45f65a43c5ea858bad080a96dcbdf",
        "0.000000,0.004179,1.000000,1.000000",
        "nutrition-table",
    ),
    (
        "26235066_5",
        "26235066",
        "26235066/5.jpg",
        1125,
        2000,
        435248,
        "d31bc4e03ea07556964d647c9d1b5292a29ba64e97b23ce47c16cbbf46dcb533",
        "0.442000,0.227556,0.793458,0.854545",
        "nutrition-table",
    ),
    (
        "3571492670004_7",
        "3571492670004",
        "357/149/267/0004/7.jpg",
        953,
        1200,
        243285,
        "125ec3aa086eee041192289492367feca403bd01cc6a27056ad317dcd7064394",
        "0.013333,0.013641,0.620000,0.956978",
        "nutrition-table",
    ),
    (
        "3256224806950_3",
        "3256224806950",
        "325/622/480/6950/3.jpg",
        788,
        491,
        110197,
        "8eea64b979e2cd300adc4ced0ecde024d9fef6687cf54c44db5abc18ef3852d5",
        "0.260692,0.040609,0.977597,0.569797",
        "nutrition-table",
    ),
    (
        "0072417153051_3",
        "0072417153051",
        "007/241/715/3051/3.jpg",
        1944,
        2592,
        1929918,
        "46c641c50efa7995e86400d34b0023d0da45070d24ce69f259ca85fad30a844a",
        "0.298586,0.266057,0.828701,0.701383;0.823117,0.395089,0.999878,0.671329",
        "nutrition-table;nutrition-table-small",
    ),
    (
        "103106_4",
        "103106",
        "103106/4.jpg",
        2434,
        1817,
        880404,
        "83700c169a4ccf16a5c7db0e71fb4fe60d67313f5655d4cd6233238580c01710",
        "0.000000,0.000000,0.999901,1.000000",
        "nutrition-table",
    ),
    (
        "20635923_2",
        "20635923",
        "20635923/2.jpg",
        2448,
        3264,
        1102005,
        "55a2b82ad9d1211a3f91d2bb737eb6a7e803923afd8af5e26968eccbbe57b679",
        "0.095565,0.558065,0.854032,0.682258",
        "nutrition-table-text",
    ),
    (
        "0072417159152_2",
        "0072417159152",
        "007/241/715/9152/2.jpg",
        2448,
        3264,
        1248922,
        "3d7eb73f60b877a2940949c85af49b711cfb69764859a3738e7f6c9195f457bf",
        "0.389458,0.120351,0.920936,0.569505;0.471405,0.925245,0.758987,1.000000;0.923077,0.240559,1.000000,0.527739",
        "nutrition-table;nutrition-table-small-energy;nutrition-table-small",
    ),
    (
        "3250390768296_4",
        "3250390768296",
        "325/039/076/8296/4.jpg",
        2000,
        1500,
        972649,
        "e37e975192ba1d8f049d2e96ba786b0bc1cf459f64ae163a67ce1135314e42d4",
        "0.532000,0.862000,0.653333,0.903500;0.647333,0.860500,0.680000,0.893500",
        "nutrition-table-small-energy;nutrition-table-small",
    ),
    (
        "0071479000013_3",
        "0071479000013",
        "007/147/900/0013/3.jpg",
        3264,
        2448,
        1845181,
        "24d5a091c551aa01404d21343529dad0e20fb3d987d6747ce9b991dc0c93b7d9",
        "0.466314,0.163282,0.886736,0.584477",
        "nutrition-table",
    ),
    (
        "0011225033674_1",
        "0011225033674",
        "001/122/503/3674/1.jpg",
        3264,
        2448,
        1825039,
        "c92ce4dcde826aed05d1d9aa814855f7c12fa8bb0d3c1e5d8504dd7db7af4697",
        "0.241830,0.004132,0.668496,0.943321",
        "nutrition-table",
    ),
    (
        "7610700946053_2",
        "7610700946053",
        "761/070/094/6053/2.jpg",
        1020,
        1360,
        346264,
        "10ace5b6fca88d180b10e01f9cc038af97cdf4480a3148203466be1f9c336388",
        "0.725000,0.275490,0.810294,0.951961",
        "nutrition-table-text",
    ),
    (
        "0064100108219_3",
        "0064100108219",
        "006/410/010/8219/3.jpg",
        2000,
        2666,
        613190,
        "b0395c01a4b58794d958f03fd1567fff7974531a1aa2d1c483197048da156300",
        "0.138785,0.179000,0.620405,0.652000",
        "nutrition-table",
    ),
    (
        "0073416006102_2",
        "0073416006102",
        "007/341/600/6102/2.jpg",
        2000,
        2666,
        414784,
        "d09cf4a32cb1a8ad1c9ae9d73a08752e9acd1a7695d3e1ce9079af9ad5ddd7e5",
        "0.225056,0.270500,0.559265,0.572500",
        "nutrition-table",
    ),
    (
        "26058450_2",
        "26058450",
        "26058450/2.jpg",
        819,
        650,
        207112,
        "e54e724b2d4c880a357b0afb927ecbf2d97b721a2e4a9ec43d9ea1f5ebf9ec8a",
        "0.000000,0.000000,1.000000,1.000000",
        "nutrition-table",
    ),
    (
        "27429464_2",
        "27429464",
        "27429464/2.jpg",
        1944,
        2592,
        1593330,
        "81ec8d7ecc5e647f01b495a96af3f3dc14b6c94d12a10bbe6eef4a61c4eb6a2a",
        "0.525849,0.172196,0.906347,0.480646",
        "nutrition-table",
    ),
    (
        "26052656_1",
        "26052656",
        "26052656/1.jpg",
        2000,
        3555,
        540007,
        "48210c9119f367bc418bd5de79aff2cab7f7cb0f24057eee140e1b330dbc8817",
        "0.776090,0.111000,0.810127,0.157000",
        "nutrition-table-small-energy",
    ),
    (
        "26036120_5",
        "26036120",
        "26036120/5.jpg",
        1560,
        2000,
        1226507,
        "b4a53be06ed5eb8f7f21471f764248ab653afe38d5c5e8f03bd264f388b18b04",
        "0.000000,0.019231,0.748500,0.988462",
        "nutrition-table",
    ),
    (
        "0042272005420_2",
        "0042272005420",
        "004/227/200/5420/2.jpg",
        1598,
        1360,
        356803,
        "46fc64d0083a6b72665a329c511720efd9c47c60f39835ace4bbbf325041dfef",
        "0.045284,0.010638,0.949277,0.995382",
        "nutrition-table",
    ),
    (
        "26191225_2",
        "26191225",
        "26191225/2.jpg",
        1125,
        2000,
        518361,
        "0bd0b5ee0e01250e6f1b4c1200d08f0d1cc0867eaafb94ae96269ac4bdd85702",
        "0.090717,0.035556,0.645976,0.874667",
        "nutrition-table",
    ),
    (
        "3250391868322_1",
        "3250391868322",
        "325/039/186/8322/1.jpg",
        2000,
        3561,
        632711,
        "3f059fce232b33e1bb3e58cc8ea80e2fcc540558cf87fa1ab3c93935fe188e63",
        "0.411401,0.083000,0.437237,0.550500",
        "nutrition-table-text",
    ),
    (
        "3222472951582_1",
        "3222472951582",
        "322/247/295/1582/1.jpg",
        3264,
        2448,
        2621579,
        "6ebf0edf2b8cfac7b3eec51a6465863e0aa7cb7b1d3ef25b6d70e734cb823e65",
        "0.403595,0.157169,0.502859,0.840380",
        "nutrition-table-text",
    ),
    (
        "0028400160148_3",
        "0028400160148",
        "002/840/016/0148/3.jpg",
        3120,
        4208,
        1852904,
        "03f3ba30d924d73f92fa101ff303fce02338bfb3dfd7d017b6db47a6e9b09751",
        "0.078422,0.260577,0.524715,0.694231",
        "nutrition-table",
    ),
    (
        "50172436_2",
        "50172436",
        "50172436/2.jpg",
        2000,
        2666,
        855873,
        "3d19028b2efbdbf8657e5e9337a589e42730b835389e8fbd8d1d01438a3a8644",
        "0.518545,0.247500,0.814430,0.753864",
        "nutrition-table",
    ),
    (
        "3564700435106_1",
        "3564700435106",
        "356/470/043/5106/1.jpg",
        1500,
        2000,
        1097943,
        "fde2ef2a5d4e7567c10dcc30228cbbead119df09ce151c9fe341fdf07efdeb30",
        "0.455000,0.033333,0.784000,0.768000",
        "nutrition-table",
    ),
    (
        "0070852000527_3",
        "0070852000527",
        "007/085/200/0527/3.jpg",
        3264,
        2448,
        1690640,
        "071428dce8f777aa61e1b3e23f613be6df7d4bf33ac61a071dc32bfd980b76e2",
        "0.300670,0.266831,0.680981,0.564658",
        "nutrition-table",
    ),
    (
        "0074333375531_2",
        "0074333375531",
        "007/433/337/5531/2.jpg",
        1125,
        2000,
        419124,
        "38210cb65b6aa213a460b9d8586b91e464bef30e02b76b192424f06e738cd89e",
        "0.015500,0.327111,0.421297,0.842810",
        "nutrition-table",
    ),
    (
        "0028400071345_2",
        "0028400071345",
        "002/840/007/1345/2.jpg",
        3264,
        2448,
        1838123,
        "40843cd14c1c274610539a458b0445c5580ebbefac377e673a7d52cf1d654b31",
        "0.301517,0.416045,0.730426,0.871762",
        "nutrition-table",
    ),
    (
        "20720162_2",
        "20720162",
        "20720162/2.jpg",
        2448,
        3264,
        1138660,
        "462439a74d913dab460af652daec4ac467696d433f864fabf5836dab1d3d2552",
        "0.122243,0.228119,0.764797,0.738544",
        "nutrition-table",
    ),
    (
        "3700003780349_1",
        "3700003780349",
        "370/000/378/0349/1.jpg",
        3087,
        2488,
        2204312,
        "a6f360b4aa545aa6739016eb0895e0aca940cceb3ad41645685088db04876e8d",
        "0.729904,0.188209,0.806270,0.796242",
        "nutrition-table-text",
    ),
    (
        "0037600106252_4",
        "0037600106252",
        "003/760/010/6252/4.jpg",
        3120,
        4208,
        2362228,
        "e0ef8fc6bca8918408425a4b3f88e2cf81379e555b5200eca7a9e801b0fc85e3",
        "0.249525,0.245513,0.668617,0.533974",
        "nutrition-table",
    ),
    (
        "0070970471254_2",
        "0070970471254",
        "007/097/047/1254/2.jpg",
        1125,
        2000,
        524563,
        "b001b96b7e0421ea6a3decf3687b0b23ecf5aae117b7b1c64e4bc0b0bb084e1f",
        "0.485918,0.375603,0.648502,0.798635;0.255484,0.159605,0.449773,0.320211",
        "nutrition-table;nutrition-table-small",
    ),
    (
        "01575118_2",
        "01575118",
        "01575118/2.jpg",
        1125,
        2000,
        468903,
        "fe5cf31b1e64d84c609c61322e328f4b7f9a6118fa5b260084e125445acea752",
        "0.315541,0.350398,0.636206,0.544194",
        "nutrition-table-text",
    ),
    (
        "0016000264694_1",
        "0016000264694",
        "001/600/026/4694/1.jpg",
        2000,
        1500,
        285622,
        "c8e0eb613ea129586d33aae5d00146d78bdce2943fe36f08e42f4859fe54ac44",
        "0.386667,0.023500,0.682000,0.768500",
        "nutrition-table-text",
    ),
    (
        "0021000653218_1",
        "0021000653218",
        "002/100/065/3218/1.jpg",
        455,
        2000,
        644198,
        "dc5d734f4f99b840312d6cbdcdd35a1f52493c2eeae4fa0349682fb0afabb5ec",
        "0.023000,0.120879,0.450500,0.835165",
        "nutrition-table",
    ),
    (
        "20840822_1",
        "20840822",
        "20840822/1.jpg",
        1125,
        2000,
        410087,
        "8446d47854e21fdeaca9d4a1ccfbd2c2dc256412a7eb1a78acb57973244c4abc",
        "0.468000,0.039111,0.733500,0.480889",
        "nutrition-table",
    ),
    (
        "0011156054502_2",
        "0011156054502",
        "001/115/605/4502/2.jpg",
        2448,
        3264,
        1131895,
        "ac9afee8bef22768404ca119827dc03c892e80ab3f1f7002164b56da7c205031",
        "0.059436,0.091912,0.639400,0.860294",
        "nutrition-table",
    ),
    (
        "0016000275348_2",
        "0016000275348",
        "001/600/027/5348/2.jpg",
        1125,
        2000,
        511106,
        "e3aa8bbf1e2abd4b615dda312f1dcebda2c57f537c0397a873405bae653007db",
        "0.058000,0.396444,0.365000,0.684444",
        "nutrition-table",
    ),
    (
        "0046100001639_2",
        "0046100001639",
        "004/610/000/1639/2.jpg",
        2988,
        5312,
        5173381,
        "4aa8d0adb67c50d695fdb2ce096d94570907c2ef0537f742641c4ee4e2a47f72",
        "0.203502,0.310241,0.534317,0.751004",
        "nutrition-table",
    ),
    (
        "0018627703211_3",
        "0018627703211",
        "001/862/770/3211/3.jpg",
        332,
        1360,
        150374,
        "f47ade12d2fbdd77b3e0b73afdb84a26a9dcc0cf35939ac623ec6d41453d4c35",
        "0.181618,0.036145,0.536029,0.972892",
        "nutrition-table",
    ),
    (
        "0073007107140_1",
        "0073007107140",
        "007/300/710/7140/1.jpg",
        3120,
        4208,
        2921589,
        "b8683ed21fbdc4f22c23885fad16f565987340b34e2a358e72d1c7b7c08343d9",
        "0.460314,0.514744,0.677281,0.749680",
        "nutrition-table",
    ),
    (
        "0036200013694_2",
        "0036200013694",
        "003/620/001/3694/2.jpg",
        3120,
        4208,
        1553624,
        "badbd8fa1eeacfea9b9c642af8d8a204d69c2485828b85fca3b9d9e39a214dd7",
        "0.360796,0.360897,0.760877,0.656510",
        "nutrition-table",
    ),
    (
        "26191218_2",
        "26191218",
        "26191218/2.jpg",
        1125,
        2000,
        533525,
        "34d180f661e918022fad563761697cee4e629b1ad6b2f6e544943302d9fef73d",
        "0.149210,0.072000,0.673280,0.852444",
        "nutrition-table",
    ),
    (
        "20574369_2",
        "20574369",
        "20574369/2.jpg",
        640,
        640,
        184489,
        "c3513d461d0eaf6f54d10fc1781187a1fb9bc428bd2caa1e2c1e2b70aaa2243b",
        "0.621875,0.510938,0.751562,0.851562",
        "nutrition-table-small",
    ),
    (
        "26167932_4",
        "26167932",
        "26167932/4.jpg",
        880,
        738,
        239860,
        "3c74bc09b35fb62fc4ef399b183b809ea6020f5366a8bcda1b9054bb56daf7d0",
        "0.000000,0.006267,1.000000,0.992011",
        "nutrition-table",
    ),
    (
        "0049000027624_2",
        "0049000027624",
        "004/900/002/7624/2.jpg",
        2000,
        1500,
        360138,
        "9f0050076a6f94d02b7fdfad867c5b5589deeae5d6d95d0a4f642f9412cd3f2c",
        "0.157470,0.032597,0.862847,0.754699",
        "nutrition-table",
    ),
    (
        "20574444_2",
        "20574444",
        "20574444/2.jpg",
        2592,
        1936,
        661531,
        "f887b9c596059aa276bd6cba09524d0d63341b23c5b524d4cfffa633d9689878",
        "0.415806,0.353009,0.770295,0.739969",
        "nutrition-table",
    ),
    (
        "0038000787270_2",
        "0038000787270",
        "003/800/078/7270/2.jpg",
        2448,
        3264,
        1019308,
        "ee129674479faacb3248212111ab1013d87960ee3178622a322745c9b16d51af",
        "0.067416,0.243873,0.582108,0.585784",
        "nutrition-table",
    ),
    (
        "0027400264993_1",
        "0027400264993",
        "002/740/026/4993/1.jpg",
        3120,
        4208,
        1962742,
        "c1031a435fda0ea9b686b53c5cee4b092f7084e8757c563db8b6a390218de308",
        "0.355038,0.166346,0.629753,0.435577",
        "nutrition-table",
    ),
    (
        "2407968021654_2",
        "2407968021654",
        "240/796/802/1654/2.jpg",
        1911,
        3302,
        1296492,
        "adf7bc130ddfa3c386b9016e8b555e403d660880636e2f26cd58a3f5883144e5",
        "0.320412,0.077446,0.383707,0.769754",
        "nutrition-table-text",
    ),
    (
        "0014054030715_2",
        "0014054030715",
        "001/405/403/0715/2.jpg",
        2000,
        2666,
        451284,
        "95d55fa77fa76ef0fe0ad579941dfdf6bb8fa82fad75e16f5f208dd670fa0fcc",
        "0.149662,0.169000,0.672543,0.740500",
        "nutrition-table",
    ),
    (
        "50300853_1",
        "50300853",
        "50300853/1.jpg",
        1944,
        2592,
        1053994,
        "b2ec37a6a066eafe95c9a9555796331441a2c8391b40cad335805fe099918995",
        "0.712191,0.353909,0.839120,0.484568",
        "nutrition-table",
    ),
    (
        "3250391868322_6",
        "3250391868322",
        "325/039/186/8322/6.jpg",
        1021,
        1360,
        322658,
        "5c07f704349e7d476c0223f29681d44237b495854e0c4a44a3b0344e75a673f9",
        "0.500000,0.061704,0.540441,0.822723",
        "nutrition-table-text",
    ),
    (
        "3410280003832_1",
        "3410280003832",
        "341/028/000/3832/1.jpg",
        3024,
        3270,
        3261686,
        "e986e16842a9f4371cf8cc16041ee742838aceca4c29b6459e9d1432b0bc0cd3",
        "0.509480,0.111772,0.807034,0.697421",
        "nutrition-table",
    ),
    (
        "3230140005024_3",
        "3230140005024",
        "323/014/000/5024/3.jpg",
        3024,
        4032,
        1457937,
        "90c8e5293feee5cc83bacd18f32859ca035b530ab4affcf52c6515f156e9132f",
        "0.410714,0.167328,0.687004,0.753968",
        "nutrition-table",
    ),
    (
        "26067674_3",
        "26067674",
        "26067674/3.jpg",
        2000,
        1125,
        289442,
        "d937845f8dfaafb85cfb1e1cc7d4390355cd1550d584c5e10e5a2227f370747b",
        "0.031111,0.447985,0.638222,0.871908",
        "nutrition-table",
    ),
    (
        "20719159_3",
        "20719159",
        "20719159/3.jpg",
        2448,
        3264,
        856874,
        "02182f1c2f0e9e46172ff73ac54e4a5cffd2c896ec72aaec1b4c739e7bab8877",
        "0.204378,0.177288,0.562938,0.815768;0.516238,0.295343,0.517770,0.297794",
        "nutrition-table;nutrition-table",
    ),
    (
        "0034000123803_7",
        "0034000123803",
        "003/400/012/3803/7.jpg",
        4208,
        3120,
        2263439,
        "235dd635ed23d93e821f3f85dd569953948afad2f2ebfb4d06e697ed2d4ce12c",
        "0.239104,0.314876,0.522044,0.709030",
        "nutrition-table",
    ),
    (
        "3222473615476_3",
        "3222473615476",
        "322/247/361/5476/3.jpg",
        2000,
        1500,
        933034,
        "34ac75f9907f3c53637077142aa10038da22cde27f875631af84454c0829abb6",
        "0.348000,0.058500,0.592000,0.946500",
        "nutrition-table-text",
    ),
    (
        "0067312002832_4",
        "0067312002832",
        "006/731/200/2832/4.jpg",
        1512,
        1533,
        855217,
        "aeedc599789b6add7aefbdbc0439d61e05b9e21944b646499bbf6578c8088edd",
        "0.140900,0.019180,0.789954,0.460317",
        "nutrition-table",
    ),
    (
        "26015637_2",
        "26015637",
        "26015637/2.jpg",
        1125,
        2000,
        451648,
        "ca56eb2bc7df1b6d6428f997cbf405ae7261066bf1b9d0149a5b798d1d226806",
        "0.236070,0.139492,0.833118,0.997645",
        "nutrition-table",
    ),
    (
        "0025616102504_2",
        "0025616102504",
        "002/561/610/2504/2.jpg",
        1125,
        2000,
        562884,
        "84096589220e1c074177915778aec2732555c4dfc8afec793610ac5c0e80fe0d",
        "0.287000,0.080000,0.846255,0.778667",
        "nutrition-table",
    ),
    (
        "20551926_2",
        "20551926",
        "20551926/2.jpg",
        3264,
        2448,
        1246335,
        "09716c32bba4f7b00b77fff227b7f62e78bedee33a8fdabaaf268479be9c4c30",
        "0.191844,0.162990,0.696078,0.815863",
        "nutrition-table",
    ),
    (
        "0054800010080_3",
        "0054800010080",
        "005/480/001/0080/3.jpg",
        3120,
        4208,
        2607798,
        "ed7569bd0aa59da0632c115f6fb90cfca8bc2b02329dbdc4b3bbe15e9cfce49a",
        "0.031743,0.334121,0.436861,0.629377",
        "nutrition-table",
    ),
    (
        "20117795_2",
        "20117795",
        "20117795/2.jpg",
        1970,
        1360,
        655546,
        "6aa3e4bed9fce3b2a9d4622cc33cd479676af70e7f0616e96ffa853b99d64bb7",
        "0.014371,0.020784,0.905147,0.960304",
        "nutrition-table",
    ),
    (
        "0043647020017_2",
        "0043647020017",
        "004/364/702/0017/2.jpg",
        1944,
        2592,
        1084683,
        "c8dc0362a3b3ecf36d3a6536abef370610865cfb3e721acb57e44caf76d866d6",
        "0.589892,0.336934,0.721836,0.606996",
        "nutrition-table",
    ),
    (
        "0043646210389_1",
        "0043646210389",
        "004/364/621/0389/1.jpg",
        3120,
        4208,
        2451229,
        "debbe1e8f795ef6641e3dfd874c64b78b2fd3f3b43ea0e1b74edab6172a9e517",
        "0.518774,0.266987,0.653517,0.681410",
        "nutrition-table-text",
    ),
    (
        "26117959_7",
        "26117959",
        "26117959/7.jpg",
        1363,
        863,
        369552,
        "1eadc776c8df97a0d32d2419681e3a6efac0a0e7dcf121a8db3587dc8d536cec",
        "0.000000,0.000000,1.000000,1.000000",
        "nutrition-table",
    ),
    (
        "20165079_2",
        "20165079",
        "20165079/2.jpg",
        2448,
        3264,
        1246787,
        "a160efe0373f4dcd2b7dc2b3e2507974b5e1d1bb62016dd9ed07fd35987b7232",
        "0.075954,0.050245,0.479565,0.903826",
        "nutrition-table",
    ),
    (
        "24632621_4",
        "24632621",
        "24632621/4.jpg",
        2448,
        3264,
        1817539,
        "0c4e43290d5c21a460acb65e0bc42c3a00a3bd59b007fd26ea88563a1699370c",
        "0.317708,0.046160,0.725490,0.978758",
        "nutrition-table",
    ),
    (
        "9300601250240_2",
        "9300601250240",
        "930/060/125/0240/2.jpg",
        1125,
        2000,
        477012,
        "b1de38fd1279143e3529f65fbd6af8a1efab06f43329fbef24a3f7df9961064e",
        "0.460500,0.102222,0.804500,0.796444",
        "nutrition-table",
    ),
    (
        "20674540_2",
        "20674540",
        "20674540/2.jpg",
        2448,
        3264,
        1051317,
        "273b324618c15cf0b6d0fee954c210221b3c35310d12608a70e945589d49a0c0",
        "0.631913,0.387992,0.846302,0.939461",
        "nutrition-table",
    ),
    (
        "0072878515276_2",
        "0072878515276",
        "007/287/851/5276/2.jpg",
        3120,
        4208,
        1674429,
        "f87f8f37a3035186b8aa684a96d5f72debe84851eb1049467cff1600a5264bb1",
        "0.437975,0.375962,0.736556,0.642555",
        "nutrition-table",
    ),
    (
        "3250390023777_3",
        "3250390023777",
        "325/039/002/3777/3.jpg",
        3072,
        1728,
        1076196,
        "ac3f1512dfaa5d1f6c9ecfcbdbd536b843be7e4f3bba1151205f4fd767facf9a",
        "0.228009,0.000000,0.927083,1.000000",
        "nutrition-table",
    ),
    (
        "0037466016450_2",
        "0037466016450",
        "003/746/601/6450/2.jpg",
        1125,
        2000,
        429456,
        "a5dc650ea2c1925d8765f71725df3c65fa32c86ca39af38e0232ac39228b1208",
        "0.432500,0.227556,0.634000,0.447111",
        "nutrition-table",
    ),
    (
        "3068320112893_9",
        "3068320112893",
        "306/832/011/2893/9.jpg",
        1749,
        1200,
        531419,
        "ec27e12db38d3180a9722d8794d826e2b4dac35fae8aba18e6f1763ab0d8b455",
        "0.002500,0.011435,0.993374,1.000000",
        "nutrition-table",
    ),
    (
        "93300292_2",
        "93300292",
        "93300292/2.jpg",
        2000,
        1125,
        220769,
        "974475b8458b485d066586b5739d6d087e75aa242eba2dbdc6a7147c1a25f4bf",
        "0.526430,0.457050,0.967308,0.690774;0.129008,0.132886,0.307979,0.260403",
        "nutrition-table;nutrition-table-small-energy",
    ),
    (
        "2000000033325_2",
        "2000000033325",
        "200/000/003/3325/2.jpg",
        1024,
        133,
        68474,
        "c6638d0ee36e3a91f671f6b37b59a1f110aa33b8f61119f8d8dbfe4968ff973d",
        "0.060150,0.005859,0.977444,0.978516",
        "nutrition-table-text",
    ),
    (
        "00854252_6",
        "00854252",
        "00854252/6.jpg",
        480,
        640,
        67778,
        "7259224bccbf5b626bc459fc6d9b91e15a0682782e4cfe0d12ef91344b99c8b9",
        "0.321875,0.214583,0.598437,0.714583",
        "nutrition-table",
    ),
    (
        "01642582_1",
        "01642582",
        "01642582/1.jpg",
        1500,
        2000,
        703939,
        "cb857fcb0500db6e36d78cd7f4ffa0749dfd6a82c805c86ba16f5bb3375897b4",
        "0.623000,0.087333,0.759000,0.531333",
        "nutrition-table-text",
    ),
    (
        "0014100074120_2",
        "0014100074120",
        "001/410/007/4120/2.jpg",
        3120,
        4208,
        2114474,
        "915bc255945939f4a49a4dbd414d3795c9016a0edbb7ee45c7f2d4d74eca1d20",
        "0.222671,0.348077,0.535646,0.683654",
        "nutrition-table",
    ),
    (
        "0038000316104_4",
        "0038000316104",
        "003/800/031/6104/4.jpg",
        3024,
        4032,
        3926416,
        "0075618e7c4e65ad4c5c596778ff96dca24c6c4a343c1711ae5f50e32a2c0740",
        "0.063244,0.076389,0.565972,0.557209",
        "nutrition-table",
    ),
    (
        "0021000419074_1",
        "0021000419074",
        "002/100/041/9074/1.jpg",
        3120,
        4208,
        2426951,
        "fc57c3cf7c80bcb0b61137d88d9f4d5e61d3265043bedc83185014088de29645",
        "0.238408,0.136285,0.457484,0.485617",
        "nutrition-table",
    ),
    (
        "3250390768296_3",
        "3250390768296",
        "325/039/076/8296/3.jpg",
        2000,
        1500,
        930040,
        "b129570ece1aea447cf4b7e102462f742e373a618074749c83213f13a411f71d",
        "0.276667,0.113500,0.858000,0.922500",
        "nutrition-table",
    ),
    (
        "20298302_2",
        "20298302",
        "20298302/2.jpg",
        2000,
        2666,
        695634,
        "ffecf25f36b6e1227c8c815644864e615227508bf119e696650d4c2385cb23ca",
        "0.067142,0.554000,0.633964,0.935000",
        "nutrition-table",
    ),
    (
        "3284230002240_2",
        "3284230002240",
        "328/423/000/2240/2.jpg",
        2000,
        1500,
        1046374,
        "e1cea36548a06a73ce77447987962540fb111d9559621ae6f8fb1c341e46cba2",
        "0.248667,0.323000,0.862667,0.672000",
        "nutrition-table",
    ),
    (
        "26155432_2",
        "26155432",
        "26155432/2.jpg",
        1125,
        2000,
        674817,
        "796872eb60b0ee496a7880b8bb4f00aac0247ac1c5a968315091e58d7a67ba38",
        "0.191391,0.038288,0.877572,0.979910",
        "nutrition-table",
    ),
    (
        "3350031653285_1",
        "3350031653285",
        "335/003/165/3285/1.jpg",
        2000,
        2697,
        363191,
        "a03bccc8ffbf39c3bfd88f77597a63ebe7c624c93b5f6d5c3373b07f0bc434a3",
        "0.572103,0.515876,0.869470,0.778876",
        "nutrition-table",
    ),
    (
        "0072869110138_3",
        "0072869110138",
        "007/286/911/0138/3.jpg",
        2000,
        2666,
        418117,
        "c20222c10e515aa3153f3038fc4acf04f0c2d16c360dd11cf134125334f3a4c1",
        "0.087188,0.168500,0.840998,0.864143",
        "nutrition-table",
    ),
    (
        "20520090_2",
        "20520090",
        "20520090/2.jpg",
        2448,
        3264,
        666912,
        "61b384f0662e1427178d3b7f767ed66f6f488b443e197d67f43f5bd1e953eb12",
        "0.353248,0.052696,0.601716,0.867239",
        "nutrition-table-text",
    ),
    (
        "20608668_3",
        "20608668",
        "20608668/3.jpg",
        2448,
        3264,
        1082691,
        "1820191bd4a4583a4516c1c27e09cb252f5b3c67cd806946be3fa5b732578218",
        "0.212916,0.001759,0.694999,0.899510",
        "nutrition-table",
    ),
    (
        "0041498000028_3",
        "0041498000028",
        "004/149/800/0028/3.jpg",
        2000,
        3555,
        602639,
        "22119a6c288b2596ae2b7300a3fa00d059fba810f707352e3704aebfac78f1c0",
        "0.187342,0.310500,0.608102,0.722916",
        "nutrition-table",
    ),
    (
        "0052159000073_2",
        "0052159000073",
        "005/215/900/0073/2.jpg",
        3264,
        2448,
        994156,
        "06c7a86d3951c1bb0fc2d3d2e24945588ed0c98095ed1e30e47cd89a589f7e9f",
        "0.157449,0.379447,0.659068,0.978778",
        "nutrition-table",
    ),
    (
        "0063667090067_4",
        "0063667090067",
        "006/366/709/0067/4.jpg",
        648,
        2000,
        602529,
        "98f4fadb0cb00d40f23eb86a8dd3fcc37d3085ebeb6365431a2ebae125e32073",
        "0.178000,0.123457,0.615500,0.899691",
        "nutrition-table",
    ),
    (
        "0021130079278_2",
        "0021130079278",
        "002/113/007/9278/2.jpg",
        3120,
        4208,
        1662367,
        "87c9799c814a59b0ec4b9b31a963c5ca9f6fa3fa0354a1c8597f4888bbcdf3f6",
        "0.537785,0.263462,0.729567,0.700962",
        "nutrition-table",
    ),
    (
        "26101989_3",
        "26101989",
        "26101989/3.jpg",
        1812,
        1640,
        1317228,
        "2031c183e50ba3b51a473da441d5e6fe2f7108325223695e745a00f414087234",
        "0.013438,0.017673,0.968876,1.000000",
        "nutrition-table",
    ),
    (
        "0016000106406_2",
        "0016000106406",
        "001/600/010/6406/2.jpg",
        3120,
        4208,
        1981516,
        "b3ec29eb53c70583d37f2fd4b6db9e3a39f2f79170c57e46289b071bd4340a45",
        "0.000000,0.338141,0.275190,0.729487",
        "nutrition-table",
    ),
    (
        "20472313_2",
        "20472313",
        "20472313/2.jpg",
        1936,
        2592,
        823642,
        "ec05e9e351b2e6dc05965cb1155343b7cce8113f058d534f2228fb9d4acf6247",
        "0.371528,0.196475,0.606531,0.718621",
        "nutrition-table",
    ),
    (
        "0016000442825_4",
        "0016000442825",
        "001/600/044/2825/4.jpg",
        3024,
        1115,
        901714,
        "ceedef64f0322591aeb0869c05c479ab777a24acf57b0c5b0169058e9afbe45e",
        "0.160538,0.016534,0.993722,0.874669",
        "nutrition-table-text",
    ),
    (
        "20889869_2",
        "20889869",
        "20889869/2.jpg",
        2448,
        3264,
        1491430,
        "71c204a69e822c5ff784d640667e2cbd058d76f88b576aead4a93725579d93e5",
        "0.137788,0.034971,0.732202,0.452222",
        "nutrition-table",
    ),
    (
        "26212630_6",
        "26212630",
        "26212630/6.jpg",
        1315,
        964,
        391990,
        "8df94f947013552e95570c4ddd3a313d5f68149160d7d2ce841ca555f8e9b099",
        "0.006224,0.000000,0.783195,0.990875",
        "nutrition-table",
    ),
    (
        "3596710458455_7",
        "3596710458455",
        "359/671/045/8455/7.jpg",
        3024,
        4032,
        1775839,
        "5578487481bfcca0874d12348b9438b173c4e71680cf3f1cf44fa8d99069f692",
        "0.620784,0.243056,0.849950,0.576720",
        "nutrition-table",
    ),
    (
        "22000279_5",
        "22000279",
        "22000279/5.jpg",
        997,
        889,
        373764,
        "9af343b85dfc0ee30d9c906a36be307a0014f26d8adf166b1f1413c8dea8e619",
        "0.000000,0.003009,1.000000,1.000000",
        "nutrition-table",
    ),
    (
        "8480000342096_3",
        "8480000342096",
        "848/000/034/2096/3.jpg",
        1017,
        1262,
        372806,
        "ac52aea0714b4d35426ee365e3b985a15410bec7cb81e799b5c7fe9e92285e20",
        "0.343899,0.273353,0.690174,0.791544",
        "nutrition-table",
    ),
    (
        "0030000059708_1",
        "0030000059708",
        "003/000/005/9708/1.jpg",
        2988,
        5312,
        3484255,
        "c22514a0c6fd24641865ed9784c0ec969db7ff301939263584a9fb206c820cab",
        "0.338291,0.130857,0.473645,0.851406",
        "nutrition-table-text",
    ),
    (
        "3257984581972_1",
        "3257984581972",
        "325/798/458/1972/1.jpg",
        1329,
        1595,
        894087,
        "6854d18a35115bfd954b99020482957e02b6d00b8adc2282138c93b0f4df9bc9",
        "0.356306,0.093303,0.600627,0.471031",
        "nutrition-table",
    ),
    (
        "0070177067731_2",
        "0070177067731",
        "007/017/706/7731/2.jpg",
        2000,
        3536,
        461087,
        "399eaab42ce9718221f5be530f6c29a7235201f8ee6fd4e489f3af698912f7d2",
        "0.688348,0.405000,0.924208,0.991000",
        "nutrition-table",
    ),
    (
        "0058449771807_2",
        "0058449771807",
        "005/844/977/1807/2.jpg",
        816,
        1360,
        260911,
        "16e8ef416bcecd1eef48e2e7a1ae1da5b527da044bb4be78a47b5a857f49679a",
        "0.025648,0.071078,0.615457,0.895833",
        "nutrition-table",
    ),
    (
        "9300601462889_2",
        "9300601462889",
        "930/060/146/2889/2.jpg",
        2000,
        1125,
        316753,
        "7ea27af4ccc1edddc6dd3576b836c952fe02b4ab063b2049999bfcf9f2bcf209",
        "0.053333,0.053000,0.941447,0.309000",
        "nutrition-table",
    ),
    (
        "0073141152327_2",
        "0073141152327",
        "007/314/115/2327/2.jpg",
        3264,
        2448,
        1977998,
        "82d2bdcd0ab607512b3a94dd9f38fd33b3f2bcc433955d44115dabfed9e0bfeb",
        "0.221578,0.255145,0.536573,0.674174",
        "nutrition-table",
    ),
    (
        "0071962226104_3",
        "0071962226104",
        "007/196/222/6104/3.jpg",
        2000,
        1500,
        343182,
        "f20050293a58040677ba7587cd5872c4021a2b414870462a9e005d25a15a1018",
        "0.343414,0.176378,0.846631,0.719249",
        "nutrition-table",
    ),
    (
        "0039000081047_4",
        "0039000081047",
        "003/900/008/1047/4.jpg",
        2448,
        3264,
        1691932,
        "c0403fb05ad062a7e8e72026ef77b6c170f5048d536ad7aad0075e539b44fddb",
        "0.252638,0.218901,0.648284,0.770815",
        "nutrition-table",
    ),
    (
        "26209142_6",
        "26209142",
        "26209142/6.jpg",
        963,
        707,
        239573,
        "538876b40d562a5de8e7fcdc73efb2c422ef596dbd10d9d38614f55940c6a1ec",
        "0.000520,0.010788,1.000000,0.906074",
        "nutrition-table",
    ),
    (
        "20511586_3",
        "20511586",
        "20511586/3.jpg",
        1936,
        2592,
        843512,
        "4835e7dbda57c9f8f50964249292a0f64b79a1027366424f9500618c208b35da",
        "0.004195,0.220041,0.921162,0.621320",
        "nutrition-table",
    ),
    (
        "0071921377601_1",
        "0071921377601",
        "007/192/137/7601/1.jpg",
        2448,
        3264,
        1442656,
        "b72e33742f9b0a5b3904a0ed0f625d8ad19e1594abd6a07632bf806430bc9918",
        "0.149816,0.087010,0.530623,0.362496",
        "nutrition-table",
    ),
)

DEFAULT_CACHE_DIR = Path("weights") / "off-nutrition"  # working-directory-relative, like the notebook
SAMPLE_SEED = 42
SAMPLE_SPLIT = {
    "train": 71,
    "validation": 24,
    "test": 24,
}  # products (119 in the corpus; two have two photographs)
MIN_RECORDS = 8
MAX_RECORDS = 2_000
MAX_BOXES = MAX_DETECTIONS  # the decoder emits at most this many boxes, so a page cannot carry more targets
MIN_BOX_SIDE = 4.0  # pixels
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def image_url(path: str) -> str:
    return f"{CORPUS_BASE_URL}{path}"


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return every pinned photograph (bytes keyed by record id) from the cache or the Open Food Facts host.

    Every file is refused on a byte-size or SHA-256 mismatch against `SAMPLE_RECORDS`.
    """
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for rid, _barcode, path, _w, _h, size, digest, _boxes, _cats in SAMPLE_RECORDS:
        local = cache / f"{rid}.jpg"
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = image_url(path)
            if fetcher is not None:
                data = fetcher(url)
            else:
                request = urllib.request.Request(
                    url, headers={"User-Agent": "dimer-table-transformer-tutorial/1.0"}
                )
                with urllib.request.urlopen(request, timeout=180) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{rid} ({path}): fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, "
                    f"pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[rid] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> list[dict[str, Any]]:
    """Decode the verified photographs into `{id, image, boxes}` records (pixel xyxy) with provenance."""
    out = []
    for rid, barcode, path, width, height, _size, _digest, boxes, cats in SAMPLE_RECORDS:
        if rid not in files:
            raise ValueError(f"corpus is missing {rid}")
        image = Image.open(io.BytesIO(files[rid]))
        image.load()
        if image.size != (width, height):
            raise ValueError(f"{rid}: served image is {image.size}, the labelled image was {(width, height)}")
        image = image.convert("RGB")
        scale = min(1.0, CORPUS_LONGEST_SIDE / max(width, height))
        if scale < 1.0:
            image = image.resize((round(width * scale), round(height * scale)), Image.LANCZOS)
        pixel_boxes = []
        kept_categories = []
        dropped = 0
        for chunk, category in zip(boxes.split(";"), cats.split(";"), strict=True):
            ymin, xmin, ymax, xmax = (float(v) for v in chunk.split(","))
            box = [xmin * image.width, ymin * image.height, xmax * image.width, ymax * image.height]
            if box[2] - box[0] < MIN_BOX_SIDE or box[3] - box[1] < MIN_BOX_SIDE:
                dropped += 1  # one annotation in the corpus is a 1.5 x 4 px sliver; the contract refuses it
                continue
            pixel_boxes.append(box)
            kept_categories.append(category)
        out.append(
            {
                "id": rid,
                "image": image,
                "boxes": pixel_boxes,
                "source_size": [width, height],
                "dropped_boxes": dropped,
                "categories": kept_categories,
                "barcode": barcode,
                "image_url": image_url(path),
            }
        )
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded draw of whole products (by barcode) into train / validation / test: `sizes` counts products."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    by_product: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        by_product.setdefault(_split_unit(record), []).append(dict(record))
    products = sorted(by_product)
    rng.shuffle(products)
    needed = sum(sizes.values())
    if len(products) < needed:
        raise ValueError(f"only {len(products)} products available, need {needed}")
    out: dict[str, list[dict[str, Any]]] = {}
    cursor = 0
    for name, count in sizes.items():
        part = [r for product in products[cursor : cursor + count] for r in by_product[product]]
        cursor += count
        rng.shuffle(part)
        out[name] = [{**r, "id": f"{name}-{i:03d}", "source_id": r["id"]} for i, r in enumerate(part)]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/image/boxes")
    for key in ("id", "image", "boxes"):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid, image, boxes = record["id"], record["image"], record["boxes"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label_name}: id must match {_ID_RE.pattern}")
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{label_name}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{label_name}: image must be a PIL.Image.Image or a file path")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE or max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(
            f"{label_name}: image side outside {MIN_IMAGE_SIDE}..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: "
            f"{image.size}"
        )
    if isinstance(boxes, str | bytes) or not isinstance(boxes, Sequence) or not 1 <= len(boxes) <= MAX_BOXES:
        raise ValueError(
            f"{label_name}: boxes must be a list of 1..{MAX_BOXES} [x_min, y_min, x_max, y_max] boxes"
        )
    checked_boxes = []
    for b, box in enumerate(boxes):
        if isinstance(box, str | bytes) or not isinstance(box, Sequence) or len(box) != 4:
            raise ValueError(f"{label_name}: boxes[{b}] must have four values")
        x0, y0, x1, y1 = (float(v) for v in box)
        if not (0 <= x0 < x1 <= width and 0 <= y0 < y1 <= height):
            raise ValueError(
                f"{label_name}: boxes[{b}] {[x0, y0, x1, y1]} must lie inside the {image.size} image "
                "with x0 < x1 and y0 < y1"
            )
        if x1 - x0 < MIN_BOX_SIDE or y1 - y0 < MIN_BOX_SIDE:
            raise ValueError(f"{label_name}: boxes[{b}] is smaller than {MIN_BOX_SIDE} px on a side")
        checked_boxes.append([x0, y0, x1, y1])
    item = {"id": rid, "image": image.convert("RGB"), "boxes": checked_boxes}
    for key in ("source_id", "categories", "barcode", "image_url", "group", "source_size", "dropped_boxes"):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a box-labelled image dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, boxes} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = [_check_record(record, index) for index, record in enumerate(records)]
    ids = [r["id"] for r in checked]
    if len(set(ids)) != len(ids):
        duplicate = next(i for i in ids if ids.count(i) > 1)
        raise ValueError(f"duplicate id {duplicate!r}")
    sides = [max(r["image"].size) for r in checked]
    n_boxes = [len(r["boxes"]) for r in checked]
    fractions = [
        (b[2] - b[0]) * (b[3] - b[1]) / (r["image"].size[0] * r["image"].size[1])
        for r in checked
        for b in r["boxes"]
    ]
    return {
        "records": checked,
        "n_records": len(checked),
        "n_boxes": sum(n_boxes),
        "boxes_per_image": {"min": min(n_boxes), "max": max(n_boxes)},
        "image_side": {"min": min(sides), "max": max(sides)},
        "box_area_fraction": {"min": round(min(fractions), 4), "max": round(max(fractions), 4)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size + bytes), so a re-encoded copy of the same photo matches."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.size[0]}x{rgb.size[1]}:".encode() + rgb.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [
        [r["id"], image_digest(r["image"]), [[round(float(v), 2) for v in b] for b in r["boxes"]]]
        for r in records
    ]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def _split_unit(record: Mapping[str, Any]) -> str:
    return str(record.get("group") or record.get("barcode") or record.get("source_id") or record["id"])


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no photograph (by decoded-pixel digest) and no product appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    units: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
            unit = _split_unit(record)
            if unit in units and units[unit] != name:
                raise ValueError(f"product {unit!r} has photographs in both {units[unit]} and {name}")
            units[unit] = name
    return {name: len(records) for name, records in splits.items()}


def split_summary(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Photographs, boxes and products per split (an observation of what the split unit was)."""
    return {
        name: {
            "images": len(records),
            "boxes": sum(len(r["boxes"]) for r in records),
            "products": len({_split_unit(r) for r in records}),
        }
        for name, records in splits.items()
    }


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test by split unit (`group` / `barcode`, else
    the image itself) after de-duplicating photographs."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    by_unit: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            by_unit.setdefault(_split_unit(record), []).append(record)
    rng = random.Random(seed)
    units = sorted(by_unit)
    rng.shuffle(units)
    n_test = max(1, round(len(units) * test_fraction))
    n_val = round(len(units) * val_fraction)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for name, chosen in (
        ("test", units[:n_test]),
        ("validation", units[n_test : n_test + n_val]),
        ("train", units[n_test + n_val :]),
    ):
        for unit in chosen:
            splits[name].extend(by_unit[unit])
    for part in splits.values():
        rng.shuffle(part)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, image, boxes}` records from a directory or a zip holding `boxes.csv` (columns `id`, `file`,
    `x_min`, `y_min`, `x_max`, `y_max`, optional `group`; one row per box, pixel coordinates) beside the image
    files; images are decoded, never extracted to disk."""
    source = Path(path)
    if source.is_dir():
        table = (source / "boxes.csv").read_text(encoding="utf-8")
        loader = lambda name: Image.open(source / name)  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist()}
        if "boxes.csv" not in members:
            raise ValueError("BYOD zip must contain boxes.csv")
        table = archive.read(members["boxes.csv"]).decode("utf-8")
        loader = lambda name: Image.open(io.BytesIO(archive.read(members[name])))  # noqa: E731
    else:
        raise ValueError("BYOD datasets must be a directory or a .zip holding boxes.csv and the image files")
    rows = list(csv.DictReader(io.StringIO(table)))
    missing = {"id", "file", "x_min", "y_min", "x_max", "y_max"} - set(rows[0].keys() if rows else set())
    if missing:
        raise ValueError(f"boxes.csv is missing columns {sorted(missing)}")
    grouped: dict[str, dict[str, Any]] = {}
    for row in rows:
        item = grouped.get(row["id"])
        if item is None:
            image = loader(row["file"])
            image.load()
            item = {"id": row["id"], "image": image.convert("RGB"), "boxes": []}
            if row.get("group"):
                item["group"] = row["group"]
            grouped[row["id"]] = item
        item["boxes"].append(
            [float(row["x_min"]), float(row["y_min"]), float(row["x_max"]), float(row["y_max"])]
        )
    return list(grouped.values())


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the boxes table of a split (one row per box, provenance) in the shape BYOD expects."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=["id", "file", "x_min", "y_min", "x_max", "y_max", "group", "category", "image_url"],
        )
        writer.writeheader()
        for record in records:
            cats = record.get("categories") or [CLASS_NAME] * len(record["boxes"])
            for box, cat in zip(record["boxes"], cats, strict=True):
                writer.writerow(
                    {
                        "id": record["id"],
                        "file": f"{record.get('source_id') or record['id']}.jpg",
                        "x_min": round(box[0], 2),
                        "y_min": round(box[1], 2),
                        "x_max": round(box[2], 2),
                        "y_max": round(box[3], 2),
                        "group": _split_unit(record),
                        "category": cat,
                        "image_url": record.get("image_url", ""),
                    }
                )
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `2357cbe2b5a5…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `TableTransformerDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "table-transformer-detection",
  "modelId": "microsoft/table-transformer-detection",
  "revision": "2357cbe2b5a5d1c03e54f32764f06058933b65ab",
  "files": [
    {
      "path": "README.md",
      "bytes": 1174,
      "sha256": "c91e7f8199313c4d24b09e73a2f6df3141268666969346cb7b94ccb59900f5dd"
    },
    {
      "path": "config.json",
      "bytes": 1228,
      "sha256": "ed5b93df2c3a59d473ddea853553a6d545d52bd4e9f8f72bf40b8a974aba4c1d"
    },
    {
      "path": "model.safetensors",
      "bytes": 115317516,
      "sha256": "8f1aa73170102c038d40155e2734b343bf07e0fe12594228a8590943b01dccf7"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 273,
      "sha256": "86a8837ae440456b0a9aef788b064921df29c20f0b67040954ce5c2fbd352c4f"
    }
  ],
  "totalBytes": 115320191
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = TableTransformerDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Referenced corpus, validation and product-level split

`fetch_corpus` downloads the 121 pinned photographs (or reads them from the cache), refuses a byte-size or SHA-256 mismatch per file before it is decoded, and `read_corpus` turns each into a `{id, image, boxes}` record — the served original downscaled to a longest side of 1,280 px, its normalised boxes rendered to pixels, one sliver annotation (1.5 × 4 px) dropped by the contract's 4 px rule — with its barcode, categories and image URL. `build_sample_dataset` draws 71 / 24 / 24 whole **products** by a seeded shuffle (72 / 24 / 25 photographs); `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no photograph (by decoded-pixel digest) and no product appears in two splits, `split_summary` reports photographs, boxes and products per split, and the training boxes table is written to `outputs/table_transformer_detection_train.csv` in the shape BYOD expects.

Look for: 121 photographs, splits 72 / 24 / 25 with 76 / 25 / 26 boxes, three digests, and four refusal probes — a duplicate id, a box outside its image, an oversized image and a record with no boxes — each rejected before `torch` does anything. About a minute on the first run for the downloads.

In [ ]:
import hashlib
import io
import json
import time

from PIL import ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
t0 = time.perf_counter()
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_count = {'byod': len(records)}
else:
    corpus = read_corpus(fetch_corpus(cache_dir='weights/off-nutrition'))
    raw_count = {'photographs': len(corpus), 'products': len({r['barcode'] for r in corpus}), 'boxes': sum(len(r['boxes']) for r in corpus), 'dropped_slivers': sum(r['dropped_boxes'] for r in corpus)}
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} ({CORPUS_RELEASE}; {CORPUS_LICENSE})'
fetch_seconds = round(time.perf_counter() - t0, 1)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
disjoint = check_split_disjoint(splits)
summary = split_summary(splits)
write_dataset_csv(train_records, 'outputs/table_transformer_detection_train.csv')
print({'data_source': data_source, 'raw': raw_count, 'splits': disjoint, 'split_summary': summary, 'fetch_seconds': fetch_seconds, 'corpus_bytes': CORPUS_BYTES})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'boxes': manifest['n_boxes'], 'boxes_per_image': manifest['boxes_per_image'], 'image_side': manifest['image_side'], 'box_area_fraction': manifest['box_area_fraction'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
print({'example': {k: example[k] for k in ('id', 'barcode', 'categories', 'image_url', 'source_size') if k in example}, 'size': example['image'].size, 'boxes': [[round(v, 1) for v in b] for b in example['boxes']]})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'box outside image': [{**train_records[0], 'boxes': [[0, 0, train_records[0]['image'].width + 5, 50]]}, *train_records[1:8]],
    'oversized image': [{**train_records[0], 'image': Image.new('RGB', (MAX_IMAGE_SIDE + 1, 8))}, *train_records[1:8]],
    'no boxes': [{**train_records[0], 'boxes': []}, *train_records[1:8]],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Detect through the inference contract, with reference boxes

Before any adaptation, the inference contract is exercised as it always was, on one test photograph. `validate_inputs` applies exactly the checks `detect` applies — image type, sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, a threshold in `[0, 1]` — and returns an input manifest; a deliberately invalid threshold is validated too and its rejection recorded as a finding. `detect` returns the PubTables queries at or above `DETECTION_THRESHOLD` with their `table` / `table rotated` labels — on a product photograph expect few or none: the checkpoint was trained on PDF page renders. Because this photograph carries reference boxes, `evaluation_report` can for the first time return **`sample-sanity`**: one `box_iou` entry per reference, the best-overlapping detection at a low threshold (0.05, so the report has something to score; the build record's probe photograph scored IoU 0.47 with one `table` detection at 0.05). A rendered preview (reference boxes in green, detections in red) is displayed. That domain gap, not a quality defect, is what the rest of the notebook adapts around.

In [ ]:
def draw_boxes(image, reference, detections, width=4):
    canvas = image.convert('RGB').copy()
    pen = ImageDraw.Draw(canvas)
    for box in reference:
        pen.rectangle([round(v) for v in box], outline=(0, 200, 0), width=width)
    for det in detections:
        pen.rectangle([round(v) for v in det['box']], outline=(230, 30, 30), width=width)
        pen.text((det['box'][0] + 4, det['box'][1] + 4), f"{det['label']} {det['score']:.2f}", fill=(230, 30, 30))
    return canvas

probe_record = test_records[0]
image = probe_record['image']
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_DETECTIONS': MAX_DETECTIONS, 'LABELS': list(LABELS), 'DETECTION_THRESHOLD': DETECTION_THRESHOLD}, 'contract': {'NUM_QUERIES': NUM_QUERIES, 'D_MODEL': D_MODEL, 'DECODER_LAYERS': DECODER_LAYERS, 'PARAMETER_COUNT': PARAMETER_COUNT}})
input_manifest = validate_inputs(image, threshold=DETECTION_THRESHOLD, names=[probe_record['id']])
try:
    validate_inputs(image, threshold=1.5)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'threshold-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/table_transformer_detection_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
started = time.perf_counter()
result = pipe.detect(image, threshold=DETECTION_THRESHOLD)
detect_seconds = round(time.perf_counter() - started, 3)
low = pipe.detect(image, threshold=0.05)
checks = {
    'at_most_num_queries': len(low['detections']) <= MAX_DETECTIONS,
    'labels_in_vocabulary': all(d['label'] in LABELS for d in low['detections']),
    'scores_descending': all(a['score'] >= b['score'] for a, b in zip(low['detections'], low['detections'][1:])),
    'boxes_inside_image': all(0 <= d['box'][0] <= d['box'][2] <= image.width + 1 and 0 <= d['box'][1] <= d['box'][3] <= image.height + 1 for d in low['detections']),
}
if not all(checks.values()):
    raise RuntimeError(f'detect output failed a sanity check: {checks}')
report = evaluation_report(low, probe_record['boxes'], sample_kind='one Open Food Facts test photograph' if not USE_BYOD else 'one BYOD test image')
print({'probe_id': probe_record['id'], 'reference_boxes': len(probe_record['boxes']), 'detections_at_threshold': len(result['detections']), 'detections_at_0.05': len(low['detections']), 'seconds': detect_seconds, 'device': pipe.device, 'checks': checks, 'findings': len(input_manifest['findings'])})
print({'verdict': report['verdict'], 'metrics': report['metrics'], 'reason': report.get('reason')})
assert report['verdict'] == 'sample-sanity'
try:
    from IPython.display import display
    display(draw_boxes(image, probe_record['boxes'], low['detections']).reduce(2))
except ImportError:
    print({'preview': 'IPython display unavailable; the preview PNG is written in Section 9'})

## 6. The fixed-box prior, the zero-shot checkpoint and the frozen policy

Three rows frame the adaptation, all on the 25 test photographs and all threshold-free: AP@0.5, AP@0.75 and mAP rank every query of every image by score, so a detector is judged on its ordering, and the recall / precision at the operating threshold (0.9, the pipeline's default) and at 0.5 are printed beside them. The **fixed-box prior** puts one box per photograph at the training split's mean normalised box — what "the table is usually here" alone buys. The **zero-shot checkpoint** (`evaluate_zero_shot`) scores each PubTables query by its `table` + `table rotated` softmax mass, the closest thing the untouched model has to the new class. The **frozen policy** is `adapt` with `trainable_layers=0`: a new two-way class head and a copy of the box head trained on the cached decoder features of the 72 training photographs under the DETR set loss for `HEAD_STEPS` full-batch steps, then scored on the test split by `evaluate`. The build record: prior AP@0.5 2.7 %, zero-shot 9.7 % (mean best IoU 0.49 — the untrained queries already sit near the tables), frozen policy 37.6 % (mAP 15.5 %) — the frozen features already locate nutrition tables once the heads know what to look for. About half a minute on CPU.

In [ ]:
HEAD_STEPS = 300  # @param {type:"integer"}
HEAD_LR = 1e-3  # @param {type:"number"}

def brief(m):
    return {'ap50': round(m['ap50'], 4), 'ap75': round(m['ap75'], 4), 'map': round(m['map'], 4), 'recall_at_0.9': round(m['operating_points'][str(DETECTION_THRESHOLD)]['recall'], 4), 'recall_at_0.5': round(m['operating_points']['0.5']['recall'], 4), 'precision_at_0.5': round(m['operating_points']['0.5']['precision'], 4), 'mean_best_iou': round(m['mean_best_iou'], 4), 'n': m['n_images']}

prior = prior_baseline(train_records, test_records, threshold=DETECTION_THRESHOLD)
print({'fixed_box_prior': brief(prior), 'baseline': prior['baseline'], 'prior_box_normalised': [round(v, 3) for v in prior['prior_box_normalised']]})
t0 = time.perf_counter()
zero_shot_test = pipe.evaluate_zero_shot(test_records)
print({'zero_shot': brief(zero_shot_test), 'policy': zero_shot_test['policy'], 'seconds': round(time.perf_counter() - t0, 1)})
t0 = time.perf_counter()
probe_result = pipe.adapt(train_records, val_records, head_steps=HEAD_STEPS, head_lr=HEAD_LR, trainable_layers=0)
frozen_test = pipe.evaluate(test_records)
print({'frozen_policy': probe_result['policy'], 'head_final_loss': round(probe_result['head_final_loss'], 4), 'validation': probe_result['history'][0]['val'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'frozen_policy_test': brief(frozen_test), 'loss': round(frozen_test['loss'], 4), 'verdict': frozen_test['verdict']})
print({'definitions': frozen_test['definitions']})
assert frozen_test['ap50'] > prior['ap50'] and probe_result['policy'] == POLICY_FROZEN

## 7. The unfrozen policy: a bounded decoder unfreeze selected against the heads

`adapt` with `TRAINABLE_LAYERS` > 0 first retrains the new heads on the frozen features (epoch 0 of the history, the frozen policy), then unfreezes the last `TRAINABLE_LAYERS` decoder layers — two by default, 3,157,504 of 28,799,431 parameters; the backbone, the input projection, the encoder, the query embeddings and the earlier decoder layers stay frozen — and trains them with both heads end to end, one photograph per step, for `EPOCHS` epochs (AdamW at `LEARNING_RATE`, weight decay 0.01, gradient clipping 0.1, seeded order, no augmentation) under the same DETR set loss: Hungarian matching of the 15 queries to the reference boxes (exact enumeration for up to four boxes), cross-entropy with a 0.1 no-object weight, L1 and GIoU box terms — all in the carried module, no external matcher. Every epoch is scored on validation, and the epoch with the **lowest validation loss** is kept — epoch 0, the heads alone, competes on equal terms, so the selected policy can be either. AP@0.5 and mAP are printed beside the loss at every epoch.

Watch the validation loss: in the build record it fell every epoch at 1e-4 (3.418 for the heads → 3.350 → 3.305 → 3.292, so epoch 3 was selected); at 3e-4 it fell only to 3.401 and epoch 3 was selected by a hair; with all six layers unfrozen it reached 3.357 at epoch 1 and rose afterwards (3.373, 3.485), so epoch 1 was kept. Note that DETR's train-mode dropout makes the per-photograph training loss sit above the full-batch head loss of epoch 0.

In [ ]:
EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
TRAINABLE_LAYERS = 2  # @param {type:"integer"}

def report_epoch(entry):
    row = {'epoch': entry['epoch'], 'stage': entry['stage'], 'train_loss': round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_loss'] = round(entry['val']['loss'], 4)
        row['val_ap50'] = round(entry['val']['ap50'], 4)
        row['val_map'] = round(entry['val']['map'], 4)
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, head_steps=HEAD_STEPS, head_lr=HEAD_LR, trainable_layers=TRAINABLE_LAYERS, epochs=EPOCHS, lr=LEARNING_RATE, progress=report_epoch)
adapt_seconds = round(time.perf_counter() - t0, 1)
report_epoch(adapt_result['history'][0])
print({'selected_policy': adapt_result['policy'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'trainable_heads': adapt_result['n_trainable_head'], 'trainable_layers': adapt_result['n_trainable_layers'], 'total_parameters': adapt_result['n_total'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test split was never used for training or policy selection, and no product in it appears in the training or validation splits. The selected model is scored exactly as the frozen policy was in Section 6, and the four rows are put side by side: fixed-box prior, zero-shot checkpoint, frozen policy, selected policy. Read the policy first: if validation kept the heads, the last two rows are the same model; if it chose the unfreeze, the delta is what the unfreeze bought on 25 photographs — the build record: 39.8 % AP@0.5 / 13.4 % AP@0.75 / 16.3 % mAP against 37.6 % / 7.4 % / 15.5 % for the heads — a small gain at 0.5 and a doubling at 0.75, i.e. the unfreeze mostly tightened boxes the heads had already found. The cell asserts the selected model beats the fixed-box prior on AP@0.5; it does **not** assert a gain over the frozen heads, because that is the question, not the answer. 25 photographs with 26 boxes from one seeded split of one corpus give no dispersion estimate — one box is about four points of recall.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
rows = {'fixed_box_prior': prior, 'zero_shot': zero_shot_test, 'frozen_policy': frozen_test, 'selected_policy': adapted_test}
comparison = {metric: {name: round(m[metric], 4) for name, m in rows.items()} for metric in ('ap50', 'ap75', 'map', 'mean_best_iou')}
comparison['recall_at_0.5'] = {name: round(m['operating_points']['0.5']['recall'], 4) for name, m in rows.items()}
comparison['precision_at_0.5'] = {name: round(m['operating_points']['0.5']['precision'], 4) for name, m in rows.items()}
comparison['recall_at_0.9'] = {name: round(m['operating_points'][str(DETECTION_THRESHOLD)]['recall'], 4) for name, m in rows.items()}
comparison['loss'] = {'frozen_policy': round(frozen_test['loss'], 4), 'selected_policy': round(adapted_test['loss'], 4)}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 4) for metric in ('ap50', 'ap75', 'map')}
comparison['delta_vs_zero_shot'] = {metric: round(adapted_test[metric] - zero_shot_test[metric], 4) for metric in ('ap50', 'ap75', 'map')}
comparison['selected_policy'] = adapt_result['policy']
for metric, row in comparison.items():
    print({metric: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'split_summary': summary,
    'single_image_report': report,
    'baselines': {'fixed_box_prior': prior, 'zero_shot': zero_shot_test},
    'frozen_policy': {'adaptation': {k: v for k, v in probe_result.items() if k not in ('history', 'trainable_names')}, 'history': probe_result['history'], 'test': frozen_test},
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/table_transformer_detection_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['ap50'] > prior['ap50']
print({'report': 'outputs/table_transformer_detection_evaluation_report.json'})

## 9. Detect before and after, export the adapter and reload it

Three test photographs are run through `detect_adapted` with the selected model at a 0.5 threshold and rendered beside the frozen policy's detections (from a fresh pipeline with new heads trained the same way — after an unfreeze the decoder inside `pipe` has moved, so the frozen column needs its own decoder) and the reference boxes: reference in green, detections in red with the head's score, which is a softmax under a 0.1 no-object weight, not a calibrated confidence; `outputs/table_transformer_detection_preview.png` holds the sheet. `detect` — the PubTables heads — still answers in its own label space on the same photographs.

`save_artifact` writes the new class head and box head and, when the unfrozen policy was selected, the trained decoder-layer tensors — about 0.5 MB for the heads alone, 13.2 MB with two decoder layers — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the class, the selected policy, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `TableTransformerDetectionPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, rebuilds the heads from the manifest, refuses any tensor that is not a decoder-layer tensor of the base, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical query scores and boxes on three photographs and an identical test AP@0.5 (VER4).

In [ ]:
import shutil

show = test_records[:3]
frozen_pipe = TableTransformerDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=pipe.device)
frozen_pipe.adapt(train_records, val_records, head_steps=HEAD_STEPS, head_lr=HEAD_LR, trainable_layers=0)
before_after = []
panels = []
for record in show:
    after = pipe.detect_adapted(record['image'], threshold=0.5)
    before = frozen_pipe.detect_adapted(record['image'], threshold=0.5)
    base = pipe.detect(record['image'], threshold=0.05)
    best_after = max((box_iou(d['box'], ref) for d in after['detections'] for ref in record['boxes']), default=0.0)
    best_before = max((box_iou(d['box'], ref) for d in before['detections'] for ref in record['boxes']), default=0.0)
    before_after.append({'id': record['id'], 'reference_boxes': len(record['boxes']), 'frozen_detections': len(before['detections']), 'frozen_best_iou': round(best_before, 4), 'selected_detections': len(after['detections']), 'selected_best_iou': round(best_after, 4), 'pubtables_detections_at_0.05': len(base['detections']), 'image_url': record.get('image_url', '')})
    print(before_after[-1])
    left = draw_boxes(record['image'], record['boxes'], before['detections'])
    right = draw_boxes(record['image'], record['boxes'], after['detections'])
    panel = Image.new('RGB', (left.width * 2 + 8, left.height), (255, 255, 255))
    panel.paste(left, (0, 0))
    panel.paste(right, (left.width + 8, 0))
    panels.append(panel)
sheet = Image.new('RGB', (max(p.width for p in panels), sum(p.height for p in panels) + 8 * (len(panels) - 1)), (255, 255, 255))
y = 0
for panel in panels:
    sheet.paste(panel, (0, y))
    y += panel.height + 8
sheet.save('outputs/table_transformer_detection_preview.png')
try:
    from IPython.display import display
    display(sheet.reduce(4))
except ImportError:
    pass

artifact_dir = Path('outputs/table_transformer_detection_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'table_transformer_detection', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'policy': artifact_manifest['adapter']['policy'], 'classes': artifact_manifest['adapter']['classes'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = TableTransformerDetectionPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
reloaded_test = reloaded.evaluate(test_records)
parity = {'queries_identical': reloaded.predict_boxes(show) == pipe.predict_boxes(show), 'ap50_in_memory': round(adapted_test['ap50'], 6), 'ap50_reloaded': round(reloaded_test['ap50'], 6), 'classes_identical': reloaded.classes == pipe.classes}
print({'reload_parity': parity, 'reloaded_policy': reloaded.adapter['policy'], 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['queries_identical'] and parity['classes_identical'] and abs(adapted_test['ap50'] - reloaded_test['ap50']) < 1e-9

weight_entry = next(entry for entry in MANIFEST['files'] if entry['path'] == WEIGHTS_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes'], 'fetched_this_run': fetched, 'weight_file': WEIGHTS_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'base_url': CORPUS_BASE_URL, 'photographs': len(SAMPLE_RECORDS), 'bytes': CORPUS_BYTES, 'license': CORPUS_LICENSE, 'class': CLASS_NAME, 'longest_side': CORPUS_LONGEST_SIDE},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'probe_id': probe_record['id'], 'single_image_report': report, 'seconds': detect_seconds},
    'comparison': comparison,
    'before_after': before_after,
    'preview_file': 'outputs/table_transformer_detection_preview.png',
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors']), 'policy': artifact_manifest['adapter']['policy']},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'timm': timm.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/table_transformer_detection_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

On 25 held-out product photographs the fixed-box prior scores 2.7 % AP@0.5, the untouched PubTables detector 9.7 %, new heads on its frozen decoder features 37.6 %, and the last two decoder layers unfrozen with them 39.8 % (mAP 16.3 %), selected at the third of three epochs by a validation loss that fell every epoch; the adapter reloads to identical query scores. That is the claim and the finding: the adaptation contract gives the PubTables detector a class and a domain it was not trained on, runs both policies end to end on a real box-labelled corpus with the set loss implemented in the open, chooses between them on validation rather than by assumption, and reports the answer against a prior and the untouched checkpoint rather than in isolation.

The test split is 25 photographs with 26 boxes from one seeded split of one small corpus with no dispersion estimate — one box is about four points of recall, so a few points of AP is noise. AP ranks every query by the new head's score; the pipeline's operating threshold of 0.9 was set for the PubTables head and the new head's scores rarely reach it (the recall at 0.9 is reported beside the recall at 0.5) — a deployment must choose its own threshold on its own labelled images, exactly as the model card says for the base detector. When the unfrozen policy is selected it changes the last decoder layers, which every query shares, so `detect` — which keeps the PubTables heads — reads a moved decoder afterwards; the artifact records which policy won.

Three things to carry to real data. **Baselines first:** the fixed-box prior and the zero-shot checkpoint on *your* images are the numbers to read before any trained head's — if the prior is competitive, your boxes are in one place and the detector is not the interesting part. **Leakage:** split by product, session or device (the contract splits by `group` / `barcode`, never by photograph). **Thresholds:** AP is threshold-free and an operating point is not; pick it on validation for the class you trained, not from the base checkpoint.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real box-labelled corpus, validate the demonstrated dataset contract without leakage, execute the inference contract with a `sample-sanity` report against reference boxes, train new detection heads and a bounded decoder unfreeze with validation-based policy selection, evaluate by AP against a prior and the zero-shot checkpoint on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, PubTables-1M accuracy, a usable acceptance threshold, or production fitness.

**Optional experiments (they do not affect the default path):** raise `TRAINABLE_LAYERS` to 6 (in the build record all six layers reached 41.0 % AP@0.5 / 18.3 % mAP but were selected at epoch 1 with the validation loss rising afterwards — a 38 MB adapter that had begun to overfit 72 photographs); raise `LEARNING_RATE` to 3e-4 (39.4 %, selected by a hair); set `EPOCHS = 0` to keep the frozen policy and read the before/after sheet as a heads-only result; raise `HEAD_STEPS`; or bring your own box-labelled images through BYOD and read the prior and the zero-shot row before either policy.

## References

- Repository README: https://github.com/kurtvalcorza/table-transformer-detection-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/table-transformer-detection-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/table-transformer-detection-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/microsoft/table-transformer-detection
- Upstream code: https://github.com/microsoft/table-transformer
- PubTables-1M: Towards Comprehensive Table Extraction From Unstructured Documents (Smock, Pesala, Abraham, 2021): https://arxiv.org/abs/2110.00061
- End-to-End Object Detection with Transformers (DETR; the set loss and Hungarian matching, Carion et al., 2020): https://arxiv.org/abs/2005.12872
- Open Food Facts nutrition-table detection dataset (boxes, ODbL) and Open Food Facts images (CC BY-SA 3.0, credited to their contributors): https://huggingface.co/datasets/openfoodfacts/nutrition-table-detection — https://world.openfoodfacts.org/data
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)